# 05 — SET50 Adaptive Fear & Alpha: Backtest Engine

ฉบับแก้ไขเพื่อป้องกันการรายงานผลที่ยังตรวจสอบไม่ได้ โดยแยก **Feature Data** ออกจาก **Execution-Price Candidate** อย่างชัดเจน และคง Backtest Gate ไว้จนกว่าจะยืนยันแหล่งราคา Corporate Actions และข้อมูล BANPU ครบถ้วน

**สถานะเริ่มต้น:** Gate ปิด (`BACKTEST_GATE_OPEN = False`) จึงไม่รัน Backtest 63 ชุดและไม่เขียนผลลัพธ์รอบใหม่


## Protocol ที่บังคับใช้

- 9 Trading Strategies × 7 Allocation Methods = 63 ชุด เมื่อ Gate เปิดเท่านั้น
- น้ำหนัก Buy ทุกวิธีต้องผ่านเพดาน `MAX_WEIGHT = 25%` ของเงินสดที่นำมาจัดสรร และน้ำหนักรวมต้องไม่เกิน 100%
- MPT ใช้ `sum(w) <= 1` เพื่ออนุญาตให้เหลือเงินสด โดยบังคับ `0 <= w_i <= 0.25` ภายใน Optimization
- Feature Data ใช้สร้าง Prediction เท่านั้น; Execution Price Candidate แยกไฟล์และใช้กำหนดราคา Order/Valuation
- ซื้อขายที่ Open ของวันถัดไป; ประเมินมูลค่าที่ Close ของวัน Execute
- Order sizing ปัดลงตาม Board Lot และต้องไม่ใช้เงินเกินงบรวมต้นทุน
- Metrics คำนวณจาก Initial Equity รวมวันเริ่มต้น ไม่ตัดผลตอบแทนวันแรกทิ้ง
- ราคาเก่าต้องถูกบันทึกเป็น Stale Valuation และไม่อนุญาตให้ใช้เกินอายุที่กำหนด


## 1. Imports, Paths และ Gate Configuration

In [131]:
import json
import math
import hashlib
import shutil
import warnings
from pathlib import Path
from itertools import product

import joblib
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform
from sklearn.covariance import LedoitWolf

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

# Locate the repository from the notebook's working directory or one of its parents.
PROJECT_ROOT = next(
    (candidate.resolve() for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "05_backtest.ipynb").is_file()
     and (candidate / "dataset").is_dir()
     and (candidate / "models").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not locate project root from working directory: {Path.cwd()}"
    )

DATASET_DIR = PROJECT_ROOT / "dataset"
BUILD_DIR = DATASET_DIR / "verified_build_20261113_152948"
MODEL_DIR = PROJECT_ROOT / "models" / "Original_Softmax_Final"

FEATURE_MASTER_PATH = BUILD_DIR / "SET50_Master_Dataset_DateSymbol_Sorted_VolZ_Fixed.parquet"
MODEL_PATH = MODEL_DIR / "final_softmax_model.joblib"
MODEL_METADATA_PATH = MODEL_DIR / "final_softmax_metadata.json"
MEMBERSHIP_PATH = DATASET_DIR / "SET50_constituents_by_halfyear_2016_2026.csv"
DECLARATIONS_PATH = DATASET_DIR / "set50_declare_dates_2016_2026.csv"

# The existing Yahoo snapshot is immutable input; all derived files live separately.
OUTPUT_DIR = PROJECT_ROOT / "backtest_results"
EXECUTION_DATA_DIR = OUTPUT_DIR / "execution_data_v2"
EXECUTION_SNAPSHOT_PATH = BUILD_DIR / "SET50_Execution_OHLCV_2026_Yahoo_auto_adjust_false_snapshot.parquet"
ORIGINAL_EXECUTION_METADATA_PATH = EXECUTION_SNAPSHOT_PATH.with_suffix(".json")
EXECUTION_ORIGINAL_COPY_PATH = EXECUTION_DATA_DIR / "execution_ohlcv_original.parquet"
EXECUTION_CORRECTED_PATH = EXECUTION_DATA_DIR / "execution_ohlcv_corrected_v2.parquet"
VERIFIED_EXECUTION_PRICE_PATH = EXECUTION_DATA_DIR / "execution_ohlcv_verified_v2.parquet"
EXECUTION_DATA_LINEAGE_PATH = OUTPUT_DIR / "execution_data_lineage.json"
VERIFIED_DATA_STATUS_PATH = OUTPUT_DIR / "verified_execution_data_status.json"
SECURITY_IDENTITY_MAPPING_PATH = OUTPUT_DIR / "security_identity_mapping.csv"
CORPORATE_ACTION_LEDGER_PATH = OUTPUT_DIR / "corporate_action_ledger_official.csv"
EXECUTION_PRICE_PATCH_PATH = DATASET_DIR / "SET_Execution_OHLCV_Source_Patch.csv"
SET_TRADING_CALENDAR_PATH = DATASET_DIR / "SET_trading_calendar_2026.csv"
OFFICIAL_EXECUTION_PRICE_PATH = BUILD_DIR / "SET50_Execution_OHLCV_2026_SET.parquet"
OFFICIAL_CORPORATE_ACTIONS_PATH = DATASET_DIR / "official_corporate_actions_2026.csv"
EXECUTION_PRICE_CANDIDATE_PATH = EXECUTION_SNAPSHOT_PATH
EXECUTION_METADATA_PATH = ORIGINAL_EXECUTION_METADATA_PATH
MODEL_DIR = PROJECT_ROOT / "models" / "Original_Softmax_Final"
FEATURE_MASTER_PATH = BUILD_DIR / "SET50_Master_Dataset_DateSymbol_Sorted_VolZ_Fixed.parquet"
MODEL_PATH = MODEL_DIR / "final_softmax_model.joblib"
MODEL_METADATA_PATH = MODEL_DIR / "final_softmax_metadata.json"
MEMBERSHIP_PATH = DATASET_DIR / "SET50_constituents_by_halfyear_2016_2026.csv"
DECLARATIONS_PATH = DATASET_DIR / "set50_declare_dates_2016_2026.csv"
ARCHIVE_MANIFEST_PATH = OUTPUT_DIR / "superseded_20261115_pre_integrity_fixes" / "archive_manifest.json"
STATUS_PATH = OUTPUT_DIR / "backtest_run_status.json"

EXPECTED_ORIGINAL_SNAPSHOT_SHA256 = "14ca82c9a481118910952cfaaaeea08daecd6548d9d99e19a66e4d9c51ef986a"
YFINANCE_REDOWNLOAD_PERFORMED = False
ORIGINAL_SNAPSHOT_MODIFIED = False
GLOBAL_EX_DATE = pd.Timestamp("2026-02-24")
GLOBAL_SHARE_FACTOR = 28 / 27
BANPU_SUSPENSION_START = pd.Timestamp("2026-07-17")
BANPU_NEW_SECURITY_START = pd.Timestamp("2026-08-04")
BANPU_FEATURES_VERIFIED_AFTER_MERGER = False

REQUIRED_INPUT_PATHS = [
    FEATURE_MASTER_PATH, MODEL_PATH, MODEL_METADATA_PATH, MEMBERSHIP_PATH,
    DECLARATIONS_PATH, EXECUTION_SNAPSHOT_PATH, ORIGINAL_EXECUTION_METADATA_PATH,
]
missing_inputs = [candidate for candidate in REQUIRED_INPUT_PATHS if not candidate.is_file()]
if missing_inputs:
    raise FileNotFoundError(
        "Required backtest input file(s) are missing:\n"
        + "\n".join(str(candidate) for candidate in missing_inputs)
    )
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXECUTION_DATA_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = ["daily_return", "momentum_20d", "vol_z", "VolRank", "CorrRank", "LiquidityRank"]
CLASS_ORDER = [0, 1, 2]  # Panic, FOMO, Normal

# สมมติฐานต้นทุนที่เลือกใช้ใน Base Case
BUY_COST_RATE = 0.0016799
SELL_COST_RATE = 0.0016799
SLIPPAGE_RATE = 0.001

INITIAL_CASH = 1_000_000.0
INITIAL_DATE = pd.Timestamp("2026-01-05")
TEST_START = pd.Timestamp("2026-01-01")
TEST_END = pd.Timestamp("2026-12-31")
BOARD_LOT = 100
MAX_WEIGHT = 0.25
SHORT_WINDOW = 60
CVaR_WINDOW = 252
CVaR_CONFIDENCE = 0.95
MIN_SHORT_HISTORY = 50
MIN_CVAR_HISTORY = 200
MPT_LAMBDA = 5.0
MAX_STALE_CALENDAR_DAYS = 5

# Gate คงปิดจนกว่าจะยืนยันครบทุกประเด็น
EXECUTION_PRICES_CONFIRMED = None
CORPORATE_ACTIONS_CONFIRMED = None
BANPU_MISSING_DATA_RESOLVED = None
TICK_AUDIT_PASSED = None
ENGINE_INTEGRITY_PASSED = None
BACKTEST_GATE_OPEN = False
ENGINE_REGRESSION_RESULTS = {}

# Tick-audit values are calculated from the loaded candidate below.
GLOBAL_TICK_AUDIT_MISMATCH_COUNT = 0

print("Gate open:", BACKTEST_GATE_OPEN)
print("Feature path exists:", FEATURE_MASTER_PATH.exists())
print("Execution candidate exists:", EXECUTION_PRICE_CANDIDATE_PATH.exists())


Gate open: False
Feature path exists: True
Execution candidate exists: True


## 2. โหลด Feature Data และโมเดล (ไม่ใช้เป็นราคา Execute)

In [132]:
feature_master = None
model = None
model_metadata = None

if FEATURE_MASTER_PATH.exists():
    feature_master = pd.read_parquet(FEATURE_MASTER_PATH)
    feature_master["Date"] = pd.to_datetime(feature_master["Date"]).dt.normalize()
    feature_master["Symbol"] = feature_master["Symbol"].astype(str).str.upper().str.strip()
    feature_master = feature_master.sort_values(["Date", "Symbol"]).reset_index(drop=True)
    print("Feature Master:", feature_master.shape)
else:
    print("ยังไม่พบ Feature Master ใน Path นี้ จึงข้ามการโหลดจริง")

if MODEL_PATH.exists():
    model = joblib.load(MODEL_PATH)
    print("Loaded model:", type(model))
else:
    print("ยังไม่พบ Model .joblib ใน Path นี้")

if MODEL_METADATA_PATH.exists():
    with open(MODEL_METADATA_PATH, "r", encoding="utf-8") as f:
        model_metadata = json.load(f)


Feature Master: (198532, 17)
Loaded model: <class 'sklearn.pipeline.Pipeline'>


## 3. ตรวจสอบ Schema และ Feature Order

In [133]:
REQUIRED_FEATURE_COLUMNS = {"Date", "Symbol", *FEATURES}

if feature_master is not None:
    missing = REQUIRED_FEATURE_COLUMNS - set(feature_master.columns)
    assert not missing, f"Feature Master ขาดคอลัมน์: {sorted(missing)}"
    assert not feature_master.duplicated(["Date", "Symbol"]).any(), "พบ Date + Symbol ซ้ำ"

if model is not None:
    if model_metadata is not None:
        meta_features = (model_metadata.get("features") or
                         model_metadata.get("feature_columns") or
                         model_metadata.get("FEATURES"))
        if meta_features is not None:
            assert list(meta_features) == FEATURES, "Feature order ใน Metadata ไม่ตรง"
    if hasattr(model, "n_features_in_"):
        assert model.n_features_in_ == len(FEATURES), "จำนวน Feature ที่โมเดลคาดหวังไม่ตรง"
    if hasattr(model, "feature_names_in_"):
        assert list(model.feature_names_in_) == FEATURES, "Feature names ของโมเดลไม่ตรง"
    assert hasattr(model, "predict") and hasattr(model, "predict_proba")

print("Schema checks completed")


Schema checks completed


## 4. โหลด Membership Events

In [134]:
membership_events = None

if not MEMBERSHIP_PATH.is_file():
    raise FileNotFoundError(f"SET50 constituents file not found: {MEMBERSHIP_PATH}")
if not DECLARATIONS_PATH.is_file():
    raise FileNotFoundError(f"SET50 effective-date file not found: {DECLARATIONS_PATH}")

constituents = pd.read_csv(MEMBERSHIP_PATH, dtype=str, keep_default_na=False)
declarations = pd.read_csv(DECLARATIONS_PATH, dtype={"year": str, "review_period": str})
required_constituent_columns = {"period"}
required_declaration_columns = {"year", "review_period", "effective_date"}
if not required_constituent_columns.issubset(constituents.columns):
    raise ValueError(f"Constituent file is missing columns: {sorted(required_constituent_columns - set(constituents.columns))}")
if not required_declaration_columns.issubset(declarations.columns):
    raise ValueError(f"Effective-date file is missing columns: {sorted(required_declaration_columns - set(declarations.columns))}")

member_columns = [column for column in constituents.columns if column.startswith("symbol_")]
if not member_columns:
    raise ValueError(f"No symbol_# columns found in {MEMBERSHIP_PATH}")
constituents["period"] = constituents["period"].astype(str).str.strip()
declarations["period"] = (
    declarations["year"].astype(str).str.strip() + "_" +
    declarations["review_period"].astype(str).str.strip().str.upper()
)
declarations["effective_date"] = pd.to_datetime(
    declarations["effective_date"], errors="raise"
).dt.normalize()
periods = constituents.merge(
    declarations[["period", "effective_date"]],
    on="period",
    how="left",
    validate="one_to_one",
)
if periods["effective_date"].isna().any():
    missing_periods = periods.loc[periods["effective_date"].isna(), "period"].tolist()
    raise ValueError(f"No official effective date found for SET50 period(s): {missing_periods}")
periods = periods.sort_values(["effective_date", "period"]).reset_index(drop=True)
if periods["effective_date"].duplicated().any():
    raise ValueError("SET50 periods contain duplicate effective dates")

membership_rows = []
previous_members = set()
for row in periods.to_dict("records"):
    current_members = {
        str(row[column]).strip().upper()
        for column in member_columns
        if str(row[column]).strip()
    }
    if len(current_members) != sum(bool(str(row[column]).strip()) for column in member_columns):
        raise ValueError(f"Duplicate constituent symbols in {row['period']}")
    effective_date = pd.Timestamp(row["effective_date"])
    membership_rows.extend(
        {"Symbol": symbol, "effective_date": effective_date, "action": "ADD"}
        for symbol in sorted(current_members - previous_members)
    )
    membership_rows.extend(
        {"Symbol": symbol, "effective_date": effective_date, "action": "REMOVE"}
        for symbol in sorted(previous_members - current_members)
    )
    previous_members = current_members

membership_events = pd.DataFrame(
    membership_rows, columns=["Symbol", "effective_date", "action"]
).sort_values(["effective_date", "Symbol"]).reset_index(drop=True)
if membership_events.empty:
    raise ValueError("No SET50 membership changes were constructed")
print("Membership periods:", len(periods))
print("Membership events:", len(membership_events))
print("First effective date:", periods["effective_date"].min())
print("Last effective date:", periods["effective_date"].max())


Membership periods: 22
Membership events: 214
First effective date: 2016-01-01 00:00:00
Last effective date: 2026-07-01 00:00:00


## 5. สร้าง Point-in-Time Membership Map

In [135]:
def build_membership_by_date(events, dates):
    # สร้าง SET50 Membership ณ แต่ละวัน โดยใช้ Effective Date
    if events is None:
        return {}
    events = events.sort_values(["effective_date", "Symbol"]).to_dict("records")
    dates = sorted(pd.to_datetime(pd.Series(dates)).dt.normalize().unique())
    state = {}
    result = {}
    pos = 0
    for current_date in dates:
        current_date = pd.Timestamp(current_date)
        while pos < len(events) and pd.Timestamp(events[pos]["effective_date"]) <= current_date:
            item = events[pos]
            state[item["Symbol"]] = (item["action"] == "ADD")
            pos += 1
        result[current_date] = {s for s, active in state.items() if active}
    return result

if feature_master is not None:
    feature_dates = pd.DatetimeIndex(sorted(feature_master["Date"].unique()))
    membership_by_date = build_membership_by_date(membership_events, feature_dates)
    for row in periods.to_dict("records"):
        effective_date = pd.Timestamp(row["effective_date"])
        position = feature_dates.searchsorted(effective_date, side="left")
        if position >= len(feature_dates):
            continue
        first_applicable_date = pd.Timestamp(feature_dates[position])
        expected_members = {
            str(row[column]).strip().upper()
            for column in member_columns
            if str(row[column]).strip()
        }
        actual_members = membership_by_date[first_applicable_date]
        assert actual_members == expected_members, (
            row["period"], first_applicable_date,
            sorted(expected_members - actual_members),
            sorted(actual_members - expected_members),
        )
    print("Effective-date membership regression check: PASS")
else:
    feature_dates = pd.DatetimeIndex([])
    membership_by_date = {}

print("Membership dates:", len(membership_by_date))


Effective-date membership regression check: PASS
Membership dates: 2638


## 6. สร้าง Feature Lookup และ Test Calendar

In [136]:
daily_feature_frames = {}
feature_test_dates = pd.DatetimeIndex([])

if feature_master is not None:
    daily_feature_frames = {
        pd.Timestamp(d): g.set_index("Symbol", drop=False)
        for d, g in feature_master.groupby("Date", sort=True)
    }
    feature_test_dates = feature_dates[
        (feature_dates >= TEST_START) & (feature_dates <= TEST_END)
    ]

next_feature_date = {
    feature_dates[i]: feature_dates[i + 1]
    for i in range(max(0, len(feature_dates) - 1))
}

print("Test feature dates:", len(feature_test_dates))


Test feature dates: 210


## 7. Prediction รายวันจาก Feature Data

In [137]:
def predict_for_date(decision_date):
    # ทำนายจาก Feature ณ วันตัดสินใจเท่านั้น ไม่ใช้ Target เป็น Input
    if feature_master is None or model is None:
        raise RuntimeError("ยังไม่ได้โหลด Feature Master และ Model")
    date = pd.Timestamp(decision_date)
    frame = daily_feature_frames.get(date)
    members = membership_by_date.get(date, set())
    if frame is None:
        return pd.DataFrame(columns=["Date", "Symbol", "Predicted_Class", "P_Panic", "P_FOMO", "P_Normal"])
    symbols = sorted(set(frame.index) & members)
    if date >= BANPU_SUSPENSION_START and not BANPU_FEATURES_VERIFIED_AFTER_MERGER:
        symbols = [symbol for symbol in symbols if symbol != "BANPU"]
    x = frame.loc[symbols, FEATURES].copy()
    x = x.loc[x.notna().all(axis=1)]
    if x.empty:
        return pd.DataFrame(columns=["Date", "Symbol", "Predicted_Class", "P_Panic", "P_FOMO", "P_Normal"])
    pred = model.predict(x)
    proba = model.predict_proba(x)
    model_classes = list(model.classes_)
    assert model_classes == CLASS_ORDER, f"Class order ไม่ตรง: {model_classes}"
    return pd.DataFrame({
        "Date": date,
        "Symbol": x.index,
        "Predicted_Class": pred,
        "P_Panic": proba[:, model_classes.index(0)],
        "P_FOMO": proba[:, model_classes.index(1)],
        "P_Normal": proba[:, model_classes.index(2)],
    }).reset_index(drop=True)


## 8. กำหนด 9 Trading Strategies

In [138]:
STRATEGIES = {
    "S1": {0: "Buy", 1: "Buy", 2: "Hold"},
    "S2": {0: "Buy", 1: "Sell", 2: "Hold"},
    "S3": {0: "Buy", 1: "Hold", 2: "Hold"},
    "S4": {0: "Sell", 1: "Buy", 2: "Hold"},
    "S5": {0: "Sell", 1: "Sell", 2: "Hold"},
    "S6": {0: "Sell", 1: "Hold", 2: "Hold"},
    "S7": {0: "Hold", 1: "Buy", 2: "Hold"},
    "S8": {0: "Hold", 1: "Sell", 2: "Hold"},
    "S9": {0: "Hold", 1: "Hold", 2: "Hold"},
}

ALLOCATION_NAMES = ["EW", "IV", "ERC", "HRP", "MinVar", "CVaR", "MPT"]
assert len(STRATEGIES) * len(ALLOCATION_NAMES) == 63

def attach_actions(prediction, strategy_name):
    out = prediction.copy()
    out["Action"] = out["Predicted_Class"].map(STRATEGIES[strategy_name])
    return out


## 9. โหลด Execution-Price Candidate แยกจาก Features

### Immutable snapshot layers (no yfinance calls)
The existing candidate has a sidecar recording `auto_adjust=False` and `actions=True`, but the original downloader cell was absent. The opt-in cell below records a reproducible request and writes a new snapshot without overwriting the frozen input. Yahoo output remains unverified as SET trade prints; this cell cannot open the Research Gate.

In [139]:
def sha256_file(file_path):
    digest = hashlib.sha256()
    with Path(file_path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


with ORIGINAL_EXECUTION_METADATA_PATH.open("r", encoding="utf-8") as stream:
    original_execution_metadata = json.load(stream)
execution_original = pd.read_parquet(EXECUTION_SNAPSHOT_PATH)
execution_original["Date"] = pd.to_datetime(execution_original["Date"], errors="raise").dt.normalize()
execution_original["Symbol"] = execution_original["Symbol"].astype(str).str.upper().str.strip()
original_hash_before = sha256_file(EXECUTION_SNAPSHOT_PATH)
if original_hash_before != EXPECTED_ORIGINAL_SNAPSHOT_SHA256:
    raise ValueError(f"Frozen snapshot hash changed; refusing to process it: {original_hash_before}")
if len(execution_original) != 17430 or execution_original["Symbol"].nunique() != 83:
    raise ValueError("Frozen snapshot dimensions changed; inspect it before rebuilding derived files")
if execution_original.duplicated(["Date", "Symbol"]).any():
    raise ValueError("Frozen snapshot contains duplicate Date/Symbol keys")

if EXECUTION_ORIGINAL_COPY_PATH.exists():
    if sha256_file(EXECUTION_ORIGINAL_COPY_PATH) != original_hash_before:
        raise ValueError("Archived original snapshot copy differs; refusing to overwrite it")
else:
    shutil.copy2(EXECUTION_SNAPSHOT_PATH, EXECUTION_ORIGINAL_COPY_PATH)
if sha256_file(EXECUTION_SNAPSHOT_PATH) != original_hash_before:
    raise AssertionError("Original snapshot changed during local processing")
if sha256_file(EXECUTION_ORIGINAL_COPY_PATH) != original_hash_before:
    raise AssertionError("Archived original snapshot copy failed hash verification")

# Candidate-only transformation from the supplied 28/27 terms; no prices are rounded.
execution_corrected = execution_original.copy(deep=True)
GLOBAL_OHLC_COLUMNS = ["Open", "High", "Low", "Close"]
global_pre_ex_mask = (
    execution_corrected["Symbol"].eq("GLOBAL")
    & execution_corrected["Date"].lt(GLOBAL_EX_DATE)
)
if not global_pre_ex_mask.any():
    raise ValueError("No pre-ex GLOBAL rows found in the frozen snapshot")
execution_corrected.loc[global_pre_ex_mask, GLOBAL_OHLC_COLUMNS] = (
    execution_corrected.loc[global_pre_ex_mask, GLOBAL_OHLC_COLUMNS]
    .apply(pd.to_numeric, errors="raise") * GLOBAL_SHARE_FACTOR
)
GLOBAL_ADJUSTED_ROW_COUNT = int(global_pre_ex_mask.sum())
GLOBAL_VOLUME_UNCHANGED = execution_corrected["Volume"].equals(execution_original["Volume"])
GLOBAL_ADJ_CLOSE_UNCHANGED = execution_corrected["Adj Close"].equals(execution_original["Adj Close"])
if not GLOBAL_VOLUME_UNCHANGED or not GLOBAL_ADJ_CLOSE_UNCHANGED:
    raise AssertionError("GLOBAL candidate must not alter Volume or Adj Close")
execution_corrected.to_parquet(EXECUTION_CORRECTED_PATH, index=False)

# Identity terms below are user-supplied and remain blocked until source URLs are attached.
identity_rows = [
    {"Security_ID": "BANPU_OLD", "Ticker": "BANPU", "Valid_From": "2026-01-05", "Valid_To": "2026-07-16", "Trading_Status": "PRE_MERGER", "Feature_Eligible": True, "Target_Security_ID": "BANPU_NEW", "Conversion_Ratio": 0.38242, "Evidence_Status": "BLOCKED_SOURCE_URL_REQUIRED"},
    {"Security_ID": "BANPU_OLD", "Ticker": "BANPUU", "Valid_From": "2026-07-17", "Valid_To": "2026-08-03", "Trading_Status": "SUSPENDED", "Feature_Eligible": False, "Target_Security_ID": "BANPU_NEW", "Conversion_Ratio": 0.38242, "Evidence_Status": "BLOCKED_SOURCE_URL_REQUIRED"},
    {"Security_ID": "BANPU_NEW", "Ticker": "BANPU", "Valid_From": "2026-08-04", "Valid_To": "2026-11-13", "Trading_Status": "NEW_IDENTITY_FEATURES_UNVERIFIED", "Feature_Eligible": False, "Target_Security_ID": "", "Conversion_Ratio": float("nan"), "Evidence_Status": "BLOCKED_SOURCE_URL_AND_FEATURE_REBUILD_REQUIRED"},
    {"Security_ID": "BPP_OLD", "Ticker": "BPP", "Valid_From": "2026-01-05", "Valid_To": "2026-07-16", "Trading_Status": "PRE_MERGER", "Feature_Eligible": True, "Target_Security_ID": "BANPU_NEW", "Conversion_Ratio": 0.80208, "Evidence_Status": "BLOCKED_SOURCE_URL_REQUIRED"},
    {"Security_ID": "BPP_OLD", "Ticker": "BPP", "Valid_From": "2026-07-17", "Valid_To": "2026-08-03", "Trading_Status": "SUSPENDED", "Feature_Eligible": False, "Target_Security_ID": "BANPU_NEW", "Conversion_Ratio": 0.80208, "Evidence_Status": "BLOCKED_SOURCE_URL_REQUIRED"},
]
security_identity_mapping = pd.DataFrame(identity_rows)
security_identity_mapping.to_csv(SECURITY_IDENTITY_MAPPING_PATH, index=False)

ORIGINAL_SNAPSHOT_MODIFIED = sha256_file(EXECUTION_SNAPSHOT_PATH) != original_hash_before
YFINANCE_REDOWNLOAD_PERFORMED = False
observed_symbols = sorted(execution_original["Symbol"].dropna().astype(str).unique().tolist())
execution_data_lineage = {
    "schema_version": "execution-data-lineage-v2",
    "source": original_execution_metadata.get("source", "UNKNOWN"),
    "auto_adjust": original_execution_metadata.get("auto_adjust", "UNKNOWN"),
    "actions": original_execution_metadata.get("actions", "UNKNOWN"),
    "interval": original_execution_metadata.get("interval", "UNKNOWN"),
    "group_by": original_execution_metadata.get("group_by", "UNKNOWN"),
    "tickers": original_execution_metadata.get("tickers", "UNKNOWN"),
    "yfinance_redownload_performed": YFINANCE_REDOWNLOAD_PERFORMED,
    "original_snapshot_modified": ORIGINAL_SNAPSHOT_MODIFIED,
    "original_snapshot": {
        "path": str(EXECUTION_SNAPSHOT_PATH), "sha256": original_hash_before,
        "rows": int(len(execution_original)), "symbol_count": int(execution_original["Symbol"].nunique()),
        "symbols_observed": observed_symbols,
        "date_min": str(execution_original["Date"].min().date()),
        "date_max": str(execution_original["Date"].max().date()),
        "columns": list(execution_original.columns),
        "provider": original_execution_metadata.get("source", "UNKNOWN"),
        "auto_adjust": original_execution_metadata.get("auto_adjust", "UNKNOWN"),
        "actions_requested": original_execution_metadata.get("actions", "UNKNOWN"),
        "request_interval": original_execution_metadata.get("interval", "UNKNOWN"),
        "request_group_by": original_execution_metadata.get("group_by", "UNKNOWN"),
        "requested_ticker_list": original_execution_metadata.get("tickers", "UNKNOWN"),
    },
    "observed_artifact_manifest": {
        "rows": int(len(execution_original)), "symbols_observed": observed_symbols,
        "date_min": str(execution_original["Date"].min().date()),
        "date_max": str(execution_original["Date"].max().date()),
        "columns": list(execution_original.columns), "sha256": original_hash_before,
        "unknown_request_fields": [
            field for field in ["interval", "group_by", "tickers"]
            if original_execution_metadata.get(field, "UNKNOWN") == "UNKNOWN"
        ],
    },
    "derived_candidate": {
        "path": str(EXECUTION_CORRECTED_PATH), "sha256": sha256_file(EXECUTION_CORRECTED_PATH),
        "basis_transformation": {
            "symbol": "GLOBAL", "effective_before": str(GLOBAL_EX_DATE.date()),
            "ohlc_factor": GLOBAL_SHARE_FACTOR, "rows_changed": GLOBAL_ADJUSTED_ROW_COUNT,
            "volume_changed": False, "adj_close_changed": False,
            "evidence_status": "USER_PROVIDED_TERMS; OFFICIAL_SOURCE_URL_NOT_ATTACHED",
        },
        "exchange_verified": False,
    },
    "verified_dataset": {
        "path": str(VERIFIED_EXECUTION_PRICE_PATH), "created": False,
        "reason": "Official/authorized source patch and official trading calendar are absent; BANPU is never forward-filled.",
        "patch_path": str(EXECUTION_PRICE_PATCH_PATH), "patch_present": EXECUTION_PRICE_PATCH_PATH.is_file(),
        "calendar_path": str(SET_TRADING_CALENDAR_PATH), "calendar_present": SET_TRADING_CALENDAR_PATH.is_file(),
    },
}
EXECUTION_METADATA_PATH = EXECUTION_DATA_LINEAGE_PATH
EXECUTION_PRICE_CANDIDATE_PATH = EXECUTION_CORRECTED_PATH
with EXECUTION_DATA_LINEAGE_PATH.open("w", encoding="utf-8") as stream:
    json.dump(execution_data_lineage, stream, ensure_ascii=False, indent=2)
verified_data_status = {
    "verified_dataset_created": False,
    "yfinance_redownload_performed": False,
    "original_snapshot_modified": ORIGINAL_SNAPSHOT_MODIFIED,
    "missing_inputs": [
        str(candidate) for candidate in [
            EXECUTION_PRICE_PATCH_PATH, SET_TRADING_CALENDAR_PATH, OFFICIAL_CORPORATE_ACTIONS_PATH
        ] if not candidate.is_file()
    ],
    "verified_output_path": str(VERIFIED_EXECUTION_PRICE_PATH),
    "blocker": "A vendor-derived candidate is not a substitute for official/authorized source patches or SET trading-status data.",
}
with VERIFIED_DATA_STATUS_PATH.open("w", encoding="utf-8") as stream:
    json.dump(verified_data_status, stream, ensure_ascii=False, indent=2)
REQUIRED_INPUT_PATHS.extend([EXECUTION_ORIGINAL_COPY_PATH, EXECUTION_CORRECTED_PATH, EXECUTION_DATA_LINEAGE_PATH])
print("Frozen snapshot copied and hash-verified:", original_hash_before)
print("GLOBAL corrected candidate rows:", GLOBAL_ADJUSTED_ROW_COUNT, "factor:", GLOBAL_SHARE_FACTOR)
print("Yahoo redownload:", YFINANCE_REDOWNLOAD_PERFORMED, "original modified:", ORIGINAL_SNAPSHOT_MODIFIED)
print("Verified execution data created:", verified_data_status["verified_dataset_created"])


Frozen snapshot copied and hash-verified: 14ca82c9a481118910952cfaaaeea08daecd6548d9d99e19a66e4d9c51ef986a
GLOBAL corrected candidate rows: 36 factor: 1.037037037037037
Yahoo redownload: False original modified: False
Verified execution data created: False


In [140]:
execution_prices = None
execution_returns = None
execution_lookup = {}
execution_metadata = execution_data_lineage

assert FEATURE_MASTER_PATH.resolve() != EXECUTION_PRICE_CANDIDATE_PATH.resolve(), (
    "Feature Master and Execution Candidate must remain separate files"
)
if not EXECUTION_CORRECTED_PATH.is_file():
    raise FileNotFoundError(f"Corrected candidate was not created: {EXECUTION_CORRECTED_PATH}")
execution_prices = pd.read_parquet(EXECUTION_CORRECTED_PATH)
execution_prices["Date"] = pd.to_datetime(execution_prices["Date"], errors="raise").dt.normalize()
execution_prices["Symbol"] = execution_prices["Symbol"].astype(str).str.upper().str.strip()
required_price_columns = {"Date", "Symbol", "Open", "High", "Low", "Close", "Volume", "Dividends", "Stock Splits"}
missing = required_price_columns - set(execution_prices.columns)
if missing:
    raise ValueError(f"Corrected execution candidate is missing columns: {sorted(missing)}")
if execution_prices.duplicated(["Date", "Symbol"]).any():
    raise ValueError("Corrected execution candidate contains duplicate Date/Symbol keys")


def security_identity_for(symbol, date):
    symbol = str(symbol).upper()
    date = pd.Timestamp(date)
    if symbol == "BANPU":
        return "BANPU_OLD" if date < BANPU_NEW_SECURITY_START else "BANPU_NEW"
    if symbol == "BPP":
        return "BPP_OLD"
    return symbol


def is_trading_eligible(row):
    date = pd.Timestamp(row.Date)
    symbol = str(row.Symbol).upper()
    values = pd.to_numeric(pd.Series([row.Open, row.High, row.Low, row.Close]), errors="coerce")
    volume = pd.to_numeric(pd.Series([row.Volume]), errors="coerce").iloc[0]
    valid_ohlc = bool(values.notna().all() and (values > 0).all())
    if not valid_ohlc or not np.isfinite(volume) or volume <= 0:
        return False
    if symbol in {"BANPU", "BPP"} and date >= BANPU_SUSPENSION_START:
        return False
    return True


execution_prices["Security_ID"] = [
    security_identity_for(row.Symbol, row.Date)
    for row in execution_prices[["Symbol", "Date"]].itertuples(index=False)
]
execution_prices["Trading_Eligible"] = [
    is_trading_eligible(row) for row in execution_prices.itertuples(index=False)
]
execution_prices["Trading_Status"] = np.where(
    execution_prices["Trading_Eligible"],
    "CANDIDATE_BAR_PRESENT_NOT_SOURCE_VERIFIED",
    "NOT_TRADABLE_OR_UNRESOLVED",
)
execution_prices = execution_prices.sort_values(["Date", "Symbol"]).reset_index(drop=True)
execution_lookup = {(row.Date, row.Symbol): row for row in execution_prices.itertuples(index=False)}
close_prices = execution_prices.pivot(index="Date", columns="Symbol", values="Close").sort_index()
tradable_mask = execution_prices.pivot(index="Date", columns="Symbol", values="Trading_Eligible").sort_index()
execution_returns = close_prices.where(tradable_mask).pct_change(fill_method=None)
print("Corrected execution candidate loaded:", execution_prices.shape)
print("Verified execution dataset available:", VERIFIED_EXECUTION_PRICE_PATH.is_file())
print("Trading-eligible candidate bars:", int(execution_prices["Trading_Eligible"].sum()))


Corrected execution candidate loaded: (17430, 13)
Verified execution dataset available: False
Trading-eligible candidate bars: 17332


In [141]:
# Read-only tick audit and GLOBAL split-basis diagnostic; candidate prices are never changed.
OHLC_COLUMNS = ["Open", "High", "Low", "Close"]
OLD_TICK_EFFECTIVE_UNTIL = pd.Timestamp("2026-11-15")
NEW_TICK_EFFECTIVE_FROM = pd.Timestamp("2026-11-16")
TICK_TOLERANCE = 2e-6
STRICT_TICK_TOLERANCE = 1e-6
SPLIT_DIAGNOSTIC_TOLERANCE = 2e-6


def required_set_tick(price, date):
    if not np.isfinite(price) or price <= 0:
        return np.nan
    if pd.Timestamp(date) < NEW_TICK_EFFECTIVE_FROM:
        bands = [(2, 0.01), (5, 0.02), (10, 0.05), (25, 0.10),
                 (100, 0.25), (200, 0.50), (400, 1.00), (np.inf, 2.00)]
    else:
        bands = [(2, 0.01), (10, 0.02), (25, 0.05), (50, 0.10),
                 (100, 0.25), (200, 0.50), (400, 1.00), (np.inf, 2.00)]
    for upper_bound, tick in bands:
        if price < upper_bound:
            return tick
    raise AssertionError("Unreachable tick-size band")


price_long = execution_prices[["Date", "Symbol", *OHLC_COLUMNS]].melt(
    id_vars=["Date", "Symbol"],
    value_vars=OHLC_COLUMNS,
    var_name="Column",
    value_name="Yahoo_Value",
).dropna(subset=["Yahoo_Value"]).copy()
price_long["Required_Tick"] = [
    required_set_tick(float(price), date)
    for price, date in zip(price_long["Yahoo_Value"], price_long["Date"])
]
price_long["Nearest_Tick_Value"] = (
    np.rint(price_long["Yahoo_Value"] / price_long["Required_Tick"])
    * price_long["Required_Tick"]
)
price_long["Deviation"] = np.abs(
    price_long["Yahoo_Value"] - price_long["Nearest_Tick_Value"]
)
tick_audit = price_long.loc[price_long["Deviation"] > TICK_TOLERANCE].copy()
tick_audit["Source"] = "GLOBAL-adjusted Yahoo snapshot candidate"
GLOBAL_TICK_AUDIT_MISMATCH_COUNT = int(tick_audit["Symbol"].eq("GLOBAL").sum())
tick_audit_path = OUTPUT_DIR / "execution_price_tick_audit.csv"
tick_audit.to_csv(tick_audit_path, index=False)

raw_price_long = execution_original[["Date", "Symbol", *OHLC_COLUMNS]].melt(
    id_vars=["Date", "Symbol"], value_vars=OHLC_COLUMNS,
    var_name="Column", value_name="Original_Value",
).dropna(subset=["Original_Value"]).copy()
raw_price_long["Required_Tick"] = [
    required_set_tick(float(price), date)
    for price, date in zip(raw_price_long["Original_Value"], raw_price_long["Date"])
]
raw_price_long["Nearest_Tick_Value"] = (
    np.rint(raw_price_long["Original_Value"] / raw_price_long["Required_Tick"])
    * raw_price_long["Required_Tick"]
)
raw_price_long["Deviation"] = np.abs(
    raw_price_long["Original_Value"] - raw_price_long["Nearest_Tick_Value"]
)
raw_tick_audit = raw_price_long.loc[raw_price_long["Deviation"] > TICK_TOLERANCE].copy()
raw_tick_audit_path = OUTPUT_DIR / "execution_price_tick_audit_original.csv"
raw_tick_audit.to_csv(raw_tick_audit_path, index=False)
GLOBAL_RAW_TICK_AUDIT_MISMATCH_COUNT = int(raw_tick_audit["Symbol"].eq("GLOBAL").sum())
GLOBAL_TICK_AUDIT_MISMATCH_COUNT = int(tick_audit["Symbol"].eq("GLOBAL").sum())

split_events_by_symbol = {}
if "Stock Splits" in execution_prices.columns:
    split_events = execution_prices.loc[
        execution_prices["Stock Splits"].notna()
        & execution_prices["Stock Splits"].ne(0),
        ["Date", "Symbol", "Stock Splits"],
    ].sort_values(["Symbol", "Date"])
    split_events_by_symbol = {
        symbol: group.reset_index(drop=True)
        for symbol, group in split_events.groupby("Symbol", sort=False)
    }


def next_vendor_split(symbol, date):
    events = split_events_by_symbol.get(symbol)
    if events is None:
        return pd.NaT, np.nan
    future = events.loc[events["Date"] > pd.Timestamp(date)]
    if future.empty:
        return pd.NaT, np.nan
    event = future.iloc[0]
    return pd.Timestamp(event["Date"]), float(event["Stock Splits"])


global_tick_diagnostic = raw_tick_audit.loc[raw_tick_audit["Symbol"].eq("GLOBAL")].copy()
corrected_long = execution_prices[["Date", "Symbol", *OHLC_COLUMNS]].melt(
    id_vars=["Date", "Symbol"], value_vars=OHLC_COLUMNS,
    var_name="Column", value_name="Corrected_Value",
)
global_tick_diagnostic = global_tick_diagnostic.merge(
    corrected_long, on=["Date", "Symbol", "Column"], how="left", validate="one_to_one"
)
global_tick_diagnostic["Corrected_Tick_Size"] = [
    required_set_tick(float(price), date) if pd.notna(price) else np.nan
    for price, date in zip(global_tick_diagnostic["Corrected_Value"], global_tick_diagnostic["Date"])
]
global_tick_diagnostic["Corrected_Nearest_Tick"] = (
    np.rint(global_tick_diagnostic["Corrected_Value"] / global_tick_diagnostic["Corrected_Tick_Size"])
    * global_tick_diagnostic["Corrected_Tick_Size"]
)
global_tick_diagnostic["Corrected_Tick_Deviation"] = np.abs(
    global_tick_diagnostic["Corrected_Value"] - global_tick_diagnostic["Corrected_Nearest_Tick"]
)
global_tick_diagnostic["Strict_Tick_Aligned"] = (
    global_tick_diagnostic["Corrected_Tick_Deviation"] <= STRICT_TICK_TOLERANCE
)
global_tick_diagnostic["Diagnostic_Tick_Aligned"] = (
    global_tick_diagnostic["Corrected_Tick_Deviation"] <= SPLIT_DIAGNOSTIC_TOLERANCE
)
next_split = [
    next_vendor_split(row.Symbol, row.Date)
    for row in global_tick_diagnostic.itertuples(index=False)
]
global_tick_diagnostic["Next_Vendor_Split_Date"] = [item[0] for item in next_split]
global_tick_diagnostic["Vendor_Split_Factor"] = [item[1] for item in next_split]
global_tick_diagnostic["Applied_Factor"] = np.where(
    global_tick_diagnostic["Date"] < GLOBAL_EX_DATE, GLOBAL_SHARE_FACTOR, 1.0
)
global_tick_diagnostic["Diagnostic_Rescaled_Price"] = global_tick_diagnostic["Corrected_Value"]
global_tick_diagnostic["Diagnostic_Tick_Size"] = global_tick_diagnostic["Corrected_Tick_Size"]
global_tick_diagnostic["Diagnostic_Nearest_Tick"] = global_tick_diagnostic["Corrected_Nearest_Tick"]
global_tick_diagnostic["Diagnostic_Tick_Deviation"] = global_tick_diagnostic["Corrected_Tick_Deviation"]
global_tick_diagnostic["Anomaly_Classification"] = (
    "GLOBAL split-basis candidate applied; strict residual checked; market basis still unverified"
)
global_tick_diagnostic["Official_Action_Confirmed"] = False
global_tick_diagnostic["Research_Execution_Eligible"] = False
global_tick_diagnostic_path = OUTPUT_DIR / "global_tick_basis_diagnostic.csv"
global_tick_diagnostic.to_csv(global_tick_diagnostic_path, index=False)

GLOBAL_TICK_DIAGNOSTIC_SUMMARY = {
    "source_mismatch_count": GLOBAL_RAW_TICK_AUDIT_MISMATCH_COUNT,
    "corrected_effective_mismatch_count": GLOBAL_TICK_AUDIT_MISMATCH_COUNT,
    "mismatch_dates": int(global_tick_diagnostic["Date"].nunique()),
    "first_mismatch_date": (
        str(global_tick_diagnostic["Date"].min().date())
        if not global_tick_diagnostic.empty else None
    ),
    "last_mismatch_date": (
        str(global_tick_diagnostic["Date"].max().date())
        if not global_tick_diagnostic.empty else None
    ),
    "vendor_split_event_date": (
        str(global_tick_diagnostic["Next_Vendor_Split_Date"].dropna().min().date())
        if global_tick_diagnostic["Next_Vendor_Split_Date"].notna().any() else None
    ),
    "vendor_split_factor": (
        float(global_tick_diagnostic["Vendor_Split_Factor"].dropna().iloc[0])
        if global_tick_diagnostic["Vendor_Split_Factor"].notna().any() else None
    ),
    "diagnostic_matches_after_rescale": int(global_tick_diagnostic["Diagnostic_Tick_Aligned"].sum()),
    "strict_matches_after_rescale": int(global_tick_diagnostic["Strict_Tick_Aligned"].sum()),
    "diagnostic_tolerance": SPLIT_DIAGNOSTIC_TOLERANCE,
    "diagnostic_max_tick_deviation": (
        float(global_tick_diagnostic["Diagnostic_Tick_Deviation"].max())
        if not global_tick_diagnostic.empty else None
    ),
    "strict_max_tick_deviation": (
        float(global_tick_diagnostic["Corrected_Tick_Deviation"].max())
        if not global_tick_diagnostic.empty else None
    ),
    "official_action_confirmed": False,
    "rescale_applied_to_execution_prices": True,
    "numerical_tolerance_only": True,
    "effective_audit_tolerance": TICK_TOLERANCE,
    "strict_diagnostic_tolerance": STRICT_TICK_TOLERANCE,
}

# Small official SET cross-check captured from the page on 2026-11-15.
# SET notes that prices are Auto Matching only; volume/value include all trading methods.
SET_GLOBAL_RECENT_SAMPLE = pd.DataFrame([
    {"Date": "2026-11-13", "SET_Open": 7.05, "SET_High": 7.10, "SET_Low": 6.90, "SET_Close": 7.10, "SET_Volume": 8360836},
    {"Date": "2026-11-12", "SET_Open": 7.05, "SET_High": 7.10, "SET_Low": 6.95, "SET_Close": 7.05, "SET_Volume": 5221816},
    {"Date": "2026-11-11", "SET_Open": 7.25, "SET_High": 7.30, "SET_Low": 6.95, "SET_Close": 7.00, "SET_Volume": 17914542},
    {"Date": "2026-11-10", "SET_Open": 7.15, "SET_High": 7.30, "SET_Low": 7.05, "SET_Close": 7.30, "SET_Volume": 7519797},
])
SET_GLOBAL_RECENT_SAMPLE["Date"] = pd.to_datetime(SET_GLOBAL_RECENT_SAMPLE["Date"])
local_global = execution_prices.loc[
    execution_prices["Symbol"].eq("GLOBAL"),
    ["Date", *OHLC_COLUMNS, "Volume"],
].rename(columns={
    "Open": "Yahoo_Open", "High": "Yahoo_High", "Low": "Yahoo_Low",
    "Close": "Yahoo_Close", "Volume": "Yahoo_Volume",
})
set_recent_crosscheck = SET_GLOBAL_RECENT_SAMPLE.merge(
    local_global, on="Date", how="left", validate="one_to_one"
)
for column in OHLC_COLUMNS:
    set_recent_crosscheck[f"{column}_Matches_SET"] = np.isclose(
        set_recent_crosscheck[f"Yahoo_{column}"],
        set_recent_crosscheck[f"SET_{column}"],
        atol=1e-6,
        rtol=0,
    )
set_recent_crosscheck["Yahoo_Minus_SET_Volume"] = (
    set_recent_crosscheck["Yahoo_Volume"] - set_recent_crosscheck["SET_Volume"]
)
set_recent_crosscheck["SET_Source"] = (
    "https://www.set.or.th/en/market/product/stock/quote/GLOBAL/historical-trading"
)
set_recent_crosscheck_path = OUTPUT_DIR / "global_set_recent_crosscheck.csv"
set_recent_crosscheck.to_csv(set_recent_crosscheck_path, index=False)

print("Tick audit rows:", len(tick_audit), "by symbol:", tick_audit.groupby("Symbol").size().to_dict())
print("GLOBAL original-source mismatches by OHLC:", global_tick_diagnostic.groupby("Column").size().to_dict())
print("GLOBAL split-basis diagnostic:", GLOBAL_TICK_DIAGNOSTIC_SUMMARY)
print("SET recent OHLC matches:", {
    column: int(set_recent_crosscheck[f"{column}_Matches_SET"].sum())
    for column in OHLC_COLUMNS
})
print("SET recent volume differences:", set_recent_crosscheck["Yahoo_Minus_SET_Volume"].tolist())
print("Diagnostic files:", tick_audit_path, global_tick_diagnostic_path, set_recent_crosscheck_path)

OHLC_VALUE_COLUMNS = ["Open", "High", "Low", "Close"]
price_values = execution_prices[OHLC_VALUE_COLUMNS].to_numpy(dtype=float)
price_finite = np.isfinite(price_values)
valid_bar_rows = execution_prices[OHLC_VALUE_COLUMNS].notna().all(axis=1)
valid_bars = execution_prices.loc[valid_bar_rows]
invalid_bar_mask = (
    (valid_bars["High"] < valid_bars[["Open", "Close", "Low"]].max(axis=1))
    | (valid_bars["Low"] > valid_bars[["Open", "Close", "High"]].min(axis=1))
)
nonpositive_price_count = int((execution_prices[OHLC_VALUE_COLUMNS] <= 0).sum().sum())
execution_manifest_complete = all(
    original_execution_metadata.get(key) not in (None, "UNKNOWN", "")
    for key in ["interval", "group_by", "tickers"]
)
price_integrity_summary = {
    "source": execution_metadata.get("source"),
    "request_manifest_complete": bool(execution_manifest_complete),
    "auto_adjust": execution_metadata.get("auto_adjust"),
    "actions_requested": execution_metadata.get("actions"),
    "rows": int(len(execution_prices)),
    "symbols": int(execution_prices["Symbol"].nunique()),
    "missing_price_values": int((~price_finite).sum()),
    "missing_by_ohlc": {column: int(execution_prices[column].isna().sum()) for column in OHLC_VALUE_COLUMNS},
    "nonpositive_price_values": nonpositive_price_count,
    "invalid_ohlc_rows": int(invalid_bar_mask.sum()),
    "duplicate_date_symbol_keys": int(execution_prices.duplicated(["Date", "Symbol"]).sum()),
    "tick_mismatch_values": int(len(tick_audit)),
    "exchange_verified": False,
    "zero_volume_or_untradable_rows": int((~execution_prices["Trading_Eligible"]).sum()),
    "original_snapshot_sha256": original_hash_before,
    "corrected_candidate_sha256": sha256_file(EXECUTION_CORRECTED_PATH),
}
execution_integrity_rows = [
    {"Check": "Provider request metadata", "Observed": f"{price_integrity_summary['source']}; auto_adjust={price_integrity_summary['auto_adjust']}; actions={price_integrity_summary['actions_requested']}", "Result": "PASS"},
    {"Check": "Exact snapshot request manifest", "Observed": f"interval={execution_metadata.get('interval')}; group_by={execution_metadata.get('group_by')}; tickers_list_present={'tickers' in execution_metadata}", "Result": "PASS" if execution_manifest_complete else "BLOCKED"},
    {"Check": "Exchange or authorized execution source", "Observed": str(price_integrity_summary["source"]), "Result": "PASS" if OFFICIAL_EXECUTION_PRICE_PATH.is_file() else "BLOCKED"},
    {"Check": "Finite Open/High/Low/Close", "Observed": str(price_integrity_summary["missing_by_ohlc"]), "Result": "PASS" if price_integrity_summary["missing_price_values"] == 0 else "FAIL"},
    {"Check": "Positive OHLC prices", "Observed": str(nonpositive_price_count), "Result": "PASS" if nonpositive_price_count == 0 else "FAIL"},
    {"Check": "OHLC range consistency", "Observed": str(price_integrity_summary["invalid_ohlc_rows"]), "Result": "PASS" if price_integrity_summary["invalid_ohlc_rows"] == 0 else "FAIL"},
    {"Check": "Unique Date/Symbol keys", "Observed": str(price_integrity_summary["duplicate_date_symbol_keys"]), "Result": "PASS" if price_integrity_summary["duplicate_date_symbol_keys"] == 0 else "FAIL"},
    {"Check": "SET tick-size conformity after candidate basis change", "Observed": f"raw_GLOBAL={GLOBAL_RAW_TICK_AUDIT_MISMATCH_COUNT}; corrected_GLOBAL_at_2e-6={GLOBAL_TICK_AUDIT_MISMATCH_COUNT}; strict_1e-6_aligned={int(global_tick_diagnostic['Strict_Tick_Aligned'].sum())}/141; verified_source={VERIFIED_EXECUTION_PRICE_PATH.is_file()}", "Result": "PASS" if tick_audit.empty and VERIFIED_EXECUTION_PRICE_PATH.is_file() else "BLOCKED"},
    {"Check": "Zero-volume bars are excluded from execution", "Observed": str(price_integrity_summary["zero_volume_or_untradable_rows"]), "Result": "PASS"},
]
execution_integrity_path = OUTPUT_DIR / "execution_price_integrity_audit.csv"
pd.DataFrame(execution_integrity_rows).to_csv(execution_integrity_path, index=False)

action_mask = execution_prices["Dividends"].fillna(0).ne(0) | execution_prices["Stock Splits"].fillna(0).ne(0)
corporate_action_vendor_events = execution_prices.loc[
    action_mask, ["Date", "Symbol", "Dividends", "Stock Splits"]
].copy()
corporate_action_vendor_events["Evidence_Source"] = str(original_execution_metadata.get("source", "UNKNOWN"))
corporate_action_vendor_events["Officially_Confirmed"] = False
corporate_action_vendor_events["Applied_By_Portfolio_Engine"] = False
corporate_action_events_path = OUTPUT_DIR / "corporate_action_vendor_events.csv"
corporate_action_vendor_events.to_csv(corporate_action_events_path, index=False)

vendor_ledger = corporate_action_vendor_events.copy()
vendor_ledger["Event_ID"] = [f"VENDOR-{i:04d}" for i in range(1, len(vendor_ledger) + 1)]
vendor_ledger["Event_Type"] = np.where(
    vendor_ledger["Stock Splits"].fillna(0).ne(0), "SPLIT",
    np.where(vendor_ledger["Dividends"].fillna(0).ne(0), "CASH_DIVIDEND", "VENDOR_EVENT"),
)
vendor_ledger["Security_ID"] = vendor_ledger["Symbol"]
vendor_ledger["Target_Security_ID"] = ""
vendor_ledger["Ex_Date"] = vendor_ledger["Date"]
for column in ["Record_Date", "Payment_Date", "Trading_Date", "Effective_Date", "Official_Source"]:
    vendor_ledger[column] = ""
vendor_ledger["Cash_Per_Share"] = vendor_ledger["Dividends"]
vendor_ledger["Share_Ratio"] = np.nan
vendor_ledger["Reconciliation_Status"] = "BLOCKED_PENDING_OFFICIAL_RECONCILIATION"
vendor_ledger["Evidence_Notes"] = "Yahoo event inventory only; not official terms and not applied by the engine."
vendor_ledger["Applied_By_Portfolio_Engine"] = False
claimed_action_rows = [
    {"Event_ID": "CLAIM-GLOBAL-20260224", "Date": "2026-02-24", "Symbol": "GLOBAL", "Event_Type": "STOCK_DIVIDEND+CASH_DIVIDEND", "Security_ID": "GLOBAL", "Target_Security_ID": "GLOBAL", "Ex_Date": "2026-02-24", "Record_Date": "2026-02-25", "Payment_Date": "2026-05-07", "Trading_Date": "2026-05-11", "Effective_Date": "2026-02-24", "Cash_Per_Share": 0.1841152263, "Share_Ratio": 1 / 27, "Official_Source": "", "Reconciliation_Status": "BLOCKED_SOURCE_URL_REQUIRED", "Evidence_Notes": "Terms supplied by user; official URL/document is not attached. Candidate OHLC factor only; not portfolio accounting."},
    {"Event_ID": "CLAIM-BANPU-20260416", "Date": "2026-04-16", "Symbol": "BANPU", "Event_Type": "CASH_DIVIDEND", "Security_ID": "BANPU_OLD", "Target_Security_ID": "", "Ex_Date": "2026-04-16", "Record_Date": "", "Payment_Date": "", "Trading_Date": "", "Effective_Date": "2026-04-16", "Cash_Per_Share": 0.18, "Share_Ratio": np.nan, "Official_Source": "", "Reconciliation_Status": "BLOCKED_REQUIRED_TERMS_AND_URL", "Evidence_Notes": "Cash amount supplied by user; record/payment terms and official URL are absent."},
    {"Event_ID": "CLAIM-BPP-20260416", "Date": "2026-04-16", "Symbol": "BPP", "Event_Type": "CASH_DIVIDEND", "Security_ID": "BPP_OLD", "Target_Security_ID": "", "Ex_Date": "2026-04-16", "Record_Date": "", "Payment_Date": "", "Trading_Date": "", "Effective_Date": "2026-04-16", "Cash_Per_Share": 0.35, "Share_Ratio": np.nan, "Official_Source": "", "Reconciliation_Status": "BLOCKED_REQUIRED_TERMS_AND_URL", "Evidence_Notes": "Cash amount supplied by user; record/payment terms and official URL are absent."},
    {"Event_ID": "CLAIM-BANPU-MERGER-20260803", "Date": "2026-08-03", "Symbol": "BANPU", "Event_Type": "MERGER", "Security_ID": "BANPU_OLD", "Target_Security_ID": "BANPU_NEW", "Ex_Date": "", "Record_Date": "", "Payment_Date": "", "Trading_Date": "2026-08-04", "Effective_Date": "2026-08-03", "Cash_Per_Share": float("nan"), "Share_Ratio": 0.38242, "Official_Source": "", "Reconciliation_Status": "BLOCKED_FRACTIONAL_SHARE_TERMS_AND_URL_REQUIRED", "Evidence_Notes": "Exchange ratio supplied by user; fractional-share settlement and source URL are absent."},
    {"Event_ID": "CLAIM-BPP-MERGER-20260803", "Date": "2026-08-03", "Symbol": "BPP", "Event_Type": "MERGER", "Security_ID": "BPP_OLD", "Target_Security_ID": "BANPU_NEW", "Ex_Date": "", "Record_Date": "", "Payment_Date": "", "Trading_Date": "2026-08-04", "Effective_Date": "2026-08-03", "Cash_Per_Share": float("nan"), "Share_Ratio": 0.80208, "Official_Source": "", "Reconciliation_Status": "BLOCKED_FRACTIONAL_SHARE_TERMS_AND_URL_REQUIRED", "Evidence_Notes": "Exchange ratio supplied by user; fractional-share settlement and source URL are absent."},
]
ledger_columns = ["Event_ID", "Date", "Symbol", "Event_Type", "Security_ID", "Target_Security_ID", "Ex_Date", "Record_Date", "Payment_Date", "Trading_Date", "Effective_Date", "Cash_Per_Share", "Share_Ratio", "Official_Source", "Reconciliation_Status", "Evidence_Notes", "Applied_By_Portfolio_Engine"]
claimed_ledger = pd.DataFrame(claimed_action_rows)
corporate_action_ledger = pd.concat(
    [vendor_ledger.reindex(columns=ledger_columns), claimed_ledger.reindex(columns=ledger_columns)],
    ignore_index=True,
)
corporate_action_ledger_path = CORPORATE_ACTION_LEDGER_PATH
corporate_action_ledger.to_csv(corporate_action_ledger_path, index=False)


def require_reconciled_corporate_actions(ledger):
    required = {"Official_Source", "Reconciliation_Status", "Applied_By_Portfolio_Engine"}
    if not required.issubset(ledger.columns):
        raise ValueError(f"Corporate-action ledger lacks required fields: {sorted(required - set(ledger.columns))}")
    source_present = ledger["Official_Source"].fillna("").astype(str).str.startswith("https://")
    reconciled = ledger["Reconciliation_Status"].eq("PASS")
    if not (source_present & reconciled).all():
        raise ValueError("Unverified corporate-action rows cannot be applied to portfolio accounting")
    return True

expected_candidate_dates = pd.DatetimeIndex(execution_prices["Date"].dropna().unique()).sort_values()
banpu_source = execution_prices.loc[execution_prices["Symbol"].eq("BANPU")].set_index("Date")
banpu_audit = pd.DataFrame({"Date": expected_candidate_dates})
banpu_audit["Candidate_Row_Present"] = banpu_audit["Date"].isin(banpu_source.index)
banpu_audit = banpu_audit.merge(
    banpu_source[OHLC_VALUE_COLUMNS + ["Volume", "Security_ID", "Trading_Eligible", "Trading_Status"]],
    left_on="Date", right_index=True, how="left", validate="one_to_one"
)
for column in OHLC_VALUE_COLUMNS:
    banpu_audit[f"Missing_{column}"] = ~np.isfinite(pd.to_numeric(banpu_audit[column], errors="coerce"))
banpu_audit["Zero_Volume_Bar"] = pd.to_numeric(banpu_audit["Volume"], errors="coerce").fillna(0).le(0)
banpu_audit["Calendar_Basis"] = "Yahoo candidate union; not an official SET calendar"
banpu_audit["Identity_Basis"] = "Provisional BANPU_OLD/BANPU_NEW mapping; official records absent"
banpu_missing_audit = banpu_audit.loc[
    ~banpu_audit["Candidate_Row_Present"]
    | banpu_audit[[f"Missing_{c}" for c in OHLC_VALUE_COLUMNS]].any(axis=1)
    | banpu_audit["Zero_Volume_Bar"]
    | ~banpu_audit["Trading_Eligible"].fillna(False)
].copy()
banpu_missing_audit_path = OUTPUT_DIR / "banpu_missing_data_audit.csv"
banpu_missing_audit.to_csv(banpu_missing_audit_path, index=False)
banpu_raw_close_dates = pd.DatetimeIndex(
    banpu_source.index[
        pd.to_numeric(banpu_source["Close"], errors="coerce").gt(0)
        & pd.to_numeric(banpu_source["Volume"], errors="coerce").gt(0)
    ]
).sort_values().unique()
banpu_eligible_close_dates = pd.DatetimeIndex(
    banpu_source.index[
        pd.to_numeric(banpu_source["Close"], errors="coerce").gt(0)
        & banpu_source["Trading_Eligible"].fillna(False)
    ]
).sort_values().unique()
banpu_missing_summary = {
    "incomplete_or_nontradable_candidate_dates": int(len(banpu_missing_audit)),
    "missing_open_dates": int(banpu_audit["Missing_Open"].sum()),
    "missing_close_dates": int(banpu_audit["Missing_Close"].sum()),
    "zero_volume_or_suspended_rows": int(banpu_audit["Zero_Volume_Bar"].sum()),
    "last_positive_volume_close_date": str(banpu_raw_close_dates[-1].date()) if len(banpu_raw_close_dates) else None,
    "last_identity_eligible_close_date": str(banpu_eligible_close_dates[-1].date()) if len(banpu_eligible_close_dates) else None,
    "trailing_stale_days_from_last_identity_eligible_close": int((expected_candidate_dates[-1] - banpu_eligible_close_dates[-1]).days) if len(banpu_eligible_close_dates) else None,
    "new_identity_execution_rows_enabled": int(
        (banpu_source["Security_ID"].eq("BANPU_NEW") & banpu_source["Trading_Eligible"].fillna(False)).sum()
    ),
    "official_set_calendar_verified": False,
    "security_identity_verified": False,
    "new_identity_features_rebuilt": False,
}

print("Execution price integrity:", price_integrity_summary)
print("BANPU missing-data summary:", banpu_missing_summary)
print("Vendor corporate-action events:", len(corporate_action_vendor_events))
print("Evidence files:", execution_integrity_path, corporate_action_events_path, corporate_action_ledger_path, banpu_missing_audit_path)
print("GLOBAL raw/corrected effective tick exceptions:", GLOBAL_RAW_TICK_AUDIT_MISMATCH_COUNT, GLOBAL_TICK_AUDIT_MISMATCH_COUNT)


Tick audit rows: 0 by symbol: {}
GLOBAL original-source mismatches by OHLC: {'Close': 35, 'High': 34, 'Low': 36, 'Open': 36}
GLOBAL split-basis diagnostic: {'source_mismatch_count': 141, 'corrected_effective_mismatch_count': 0, 'mismatch_dates': 36, 'first_mismatch_date': '2026-01-05', 'last_mismatch_date': '2026-02-23', 'vendor_split_event_date': '2026-02-24', 'vendor_split_factor': 1.037037037037037, 'diagnostic_matches_after_rescale': 141, 'strict_matches_after_rescale': 125, 'diagnostic_tolerance': 2e-06, 'diagnostic_max_tick_deviation': 1.1161521635827398e-06, 'strict_max_tick_deviation': 1.1161521635827398e-06, 'official_action_confirmed': False, 'rescale_applied_to_execution_prices': True, 'numerical_tolerance_only': True, 'effective_audit_tolerance': 2e-06, 'strict_diagnostic_tolerance': 1e-06}
SET recent OHLC matches: {'Open': 4, 'High': 4, 'Low': 4, 'Close': 4}
SET recent volume differences: [-436.0, -416.0, -242.0, -59697.0]
Diagnostic files: D:\Project_Github\set_fear_index

## GLOBAL Price-Basis Candidate and Verification Limits

- yfinance documents `auto_adjust` as applying adjustments to all OHLC fields. Setting it to `False` disables that yfinance operation; it does not certify that Yahoo's historical series equals exchange trade prints. [yfinance `download` reference](https://ranaroussi.github.io/yfinance/reference/api/yfinance.download.html)
- The SET schedule for Jan 5?Feb 23, 2026 is the pre-November-16 schedule. For GLOBAL prices in the 5?10 THB band, its tick is 0.05 THB. The [older SET tick-size page](https://www.set.or.th/en/trading-units-tick-sizes-price-limits) lists that schedule; the [new schedule](https://www.set.or.th/en/market/information/trading-procedure/price-bands-and-tick-sizes) is effective Nov 16, 2026.
- The candidate audit finds 141 values (36 days: Open 36, High 34, Low 36, Close 35), all GLOBAL, from 2026-01-05 through 2026-02-23. Yahoo reports a 1.037037... split factor on 2026-02-24 (= 28/27). Multiplying these historical candidate values by that factor places all 141 within 2e-6 THB of the 0.05 tick grid (maximum deviation about 1.12e-6). This is strong evidence that the earlier values are on a vendor split-adjusted basis, not 141 independent quote errors. It is not official confirmation of the corporate action or an exact reconstruction of trade prints.
- This calculation is diagnostic only: it does not round, replace, or otherwise alter the OHLCV candidate. The derived diagnostic values are not passed into the portfolio engine. Both [`global_tick_basis_diagnostic.csv`](./global_tick_basis_diagnostic.csv) and the original-value [`execution_price_tick_audit.csv`](./execution_price_tick_audit.csv) are saved separately.
- A small current-price cross-check against the [SET GLOBAL Historical Trading page](https://www.set.or.th/en/market/product/stock/quote/GLOBAL/historical-trading) matches Yahoo's OHLC for Nov 10?13, 2026. The displayed volumes differ; SET says its prices are Auto Matching only and its volume/value include all trading methods. That recent subset does not verify Jan?Feb data. The SET page states ?Back to 6 months,? so the Jan?Feb observations are outside the public page's current visible range as of Nov 15, 2026.
- **Decision:** retain Yahoo data unchanged as a candidate, classify GLOBAL's pre-event rows as *split-basis suspicious / unresolved*, and keep `EXECUTION_PRICES_CONFIRMED` and `TICK_AUDIT_PASSED` false. Do not use the 28/27 calculation as a trading-price repair. Obtain exchange/authorized historical records covering the pre-event period and confirm the Feb 24 rights/benefits event before clearing this blocker.


## 10. Allocation Helpers — บังคับ Cap ทุกครั้ง

In [142]:
def apply_weight_cap(weights, cap=MAX_WEIGHT):
    # จำกัดน้ำหนักต่อหุ้นและกระจายน้ำหนักส่วนเกินซ้ำ; ส่วนที่เกินความจุคงเป็นเงินสด
    w = pd.Series(weights, dtype=float).replace([np.inf, -np.inf], np.nan).fillna(0).clip(lower=0)
    if w.empty or w.sum() <= 0:
        return pd.Series(0.0, index=w.index)
    if not (0 < cap <= 1):
        raise ValueError("cap ต้องอยู่ระหว่าง 0 ถึง 1")
    w = w / w.sum()
    output = pd.Series(0.0, index=w.index)
    remaining_names = list(w.index)
    remaining_budget = 1.0
    while remaining_names and remaining_budget > 1e-12:
        base = w.loc[remaining_names]
        if base.sum() <= 0:
            break
        proposed = base / base.sum() * remaining_budget
        over = proposed[proposed > cap + 1e-12]
        if over.empty:
            output.loc[remaining_names] = proposed
            remaining_budget = 0.0
            break
        for symbol in over.index:
            output.loc[symbol] = cap
            remaining_budget -= cap
            remaining_names.remove(symbol)
    return output.clip(lower=0, upper=cap)


def equal_weights(symbols):
    symbols = list(symbols)
    return pd.Series(1.0 / len(symbols), index=symbols) if symbols else pd.Series(dtype=float)


def finalize_allocation(raw_weights):
    # ทุกวิธีต้องผ่าน cap รวมถึงกรณีมี Buy เพียงหุ้นเดียว
    return apply_weight_cap(raw_weights, MAX_WEIGHT)


def get_complete_history(symbols, decision_date, window, min_history):
    if execution_returns is None or not symbols:
        return None
    history = execution_returns.loc[
        execution_returns.index < pd.Timestamp(decision_date), list(symbols)
    ].tail(window).dropna(axis=0, how="any")
    return history if len(history) >= min_history else None


## 11. Allocation Methods — EW, IV

In [143]:
def allocation_ew(symbols, decision_date):
    return finalize_allocation(equal_weights(symbols)), "OK"


def allocation_iv(symbols, decision_date):
    history = get_complete_history(symbols, decision_date, SHORT_WINDOW, MIN_SHORT_HISTORY)
    if history is None:
        return finalize_allocation(equal_weights(symbols)), "EW_FALLBACK"
    vol = history.std(ddof=1).replace(0, np.nan)
    if vol.isna().any():
        return finalize_allocation(equal_weights(symbols)), "EW_FALLBACK"
    raw = 1.0 / vol
    return finalize_allocation(raw), "OK"


## 12. Allocation Methods — ERC, HRP, MinVar

In [144]:
def ledoit_wolf_cov(history):
    lw = LedoitWolf().fit(history.values)
    return pd.DataFrame(lw.covariance_, index=history.columns, columns=history.columns)


def allocation_erc(symbols, decision_date):
    history = get_complete_history(symbols, decision_date, SHORT_WINDOW, MIN_SHORT_HISTORY)
    if history is None:
        return finalize_allocation(equal_weights(symbols)), "EW_FALLBACK"
    cov = ledoit_wolf_cov(history).values
    n = len(symbols)
    if n == 1:
        raw = pd.Series([1.0], index=symbols)
    else:
        def objective(w):
            variance = float(w @ cov @ w)
            rc = w * (cov @ w)
            return float(np.sum((rc - variance / n) ** 2))
        opt = minimize(objective, np.repeat(1/n, n), method="SLSQP",
                       bounds=[(0, 1)] * n,
                       constraints=[{"type": "eq", "fun": lambda w: np.sum(w) - 1}],
                       options={"maxiter": 500, "ftol": 1e-12})
        if not opt.success:
            return finalize_allocation(equal_weights(symbols)), "EW_FALLBACK"
        raw = pd.Series(opt.x, index=symbols)
    return finalize_allocation(raw), "OK"


def allocation_hrp(symbols, decision_date):
    history = get_complete_history(symbols, decision_date, SHORT_WINDOW, MIN_SHORT_HISTORY)
    if history is None:
        return finalize_allocation(equal_weights(symbols)), "EW_FALLBACK"
    if len(symbols) == 1:
        return finalize_allocation(pd.Series([1.0], index=symbols)), "OK"
    try:
        corr = history.corr().clip(-1, 1)
        distance = np.sqrt(np.maximum(0, (1 - corr) / 2))
        np.fill_diagonal(distance.values, 0)
        tree = linkage(squareform(distance.values, checks=False), method="single")
        ordered = list(corr.index[leaves_list(tree)])
        cov = ledoit_wolf_cov(history[ordered])
        weights = pd.Series(1.0, index=ordered)
        clusters = [ordered]
        while clusters:
            next_clusters = []
            for cluster in clusters:
                if len(cluster) <= 1:
                    continue
                split = len(cluster) // 2
                left, right = cluster[:split], cluster[split:]
                c_left, c_right = cov.loc[left, left], cov.loc[right, right]
                inv_left = 1 / np.diag(c_left.values)
                inv_right = 1 / np.diag(c_right.values)
                wl, wr = inv_left / inv_left.sum(), inv_right / inv_right.sum()
                vl = float(wl @ c_left.values @ wl)
                vr = float(wr @ c_right.values @ wr)
                alpha = 1 - vl / (vl + vr)
                weights[left] *= alpha
                weights[right] *= (1 - alpha)
                next_clusters.extend([left, right])
            clusters = next_clusters
        return finalize_allocation(weights.reindex(symbols)), "OK"
    except Exception:
        return finalize_allocation(equal_weights(symbols)), "EW_FALLBACK"


def allocation_minvar(symbols, decision_date):
    history = get_complete_history(symbols, decision_date, SHORT_WINDOW, MIN_SHORT_HISTORY)
    if history is None:
        return finalize_allocation(equal_weights(symbols)), "EW_FALLBACK"
    cov = ledoit_wolf_cov(history).values
    n = len(symbols)
    if n == 1:
        raw = pd.Series([1.0], index=symbols)
    else:
        opt = minimize(lambda w: float(w @ cov @ w), np.repeat(1/n, n), method="SLSQP",
                       bounds=[(0, 1)] * n,
                       constraints=[{"type": "eq", "fun": lambda w: np.sum(w) - 1}],
                       options={"maxiter": 500, "ftol": 1e-12})
        if not opt.success:
            return finalize_allocation(equal_weights(symbols)), "EW_FALLBACK"
        raw = pd.Series(opt.x, index=symbols)
    return finalize_allocation(raw), "OK"


## 13. Allocation Methods — CVaR และ MPT (Allow Cash)

In [145]:
def allocation_cvar(symbols, decision_date):
    history = get_complete_history(symbols, decision_date, CVaR_WINDOW, MIN_CVAR_HISTORY)
    if history is None:
        return finalize_allocation(equal_weights(symbols)), "EW_FALLBACK"
    n = len(symbols)
    if n == 1:
        raw = pd.Series([1.0], index=symbols)
    else:
        returns = history.values
        tail_count = max(1, int(np.ceil((1 - CVaR_CONFIDENCE) * len(returns))))
        def objective(w):
            losses = -(returns @ w)
            return float(np.mean(np.sort(losses)[-tail_count:]))
        opt = minimize(objective, np.repeat(1/n, n), method="SLSQP",
                       bounds=[(0, 1)] * n,
                       constraints=[{"type": "eq", "fun": lambda w: np.sum(w) - 1}],
                       options={"maxiter": 500, "ftol": 1e-10})
        if not opt.success:
            return finalize_allocation(equal_weights(symbols)), "EW_FALLBACK"
        raw = pd.Series(opt.x, index=symbols)
    return finalize_allocation(raw), "OK"


def optimize_mpt(mu, covariance, risk_aversion=MPT_LAMBDA, cap=MAX_WEIGHT):
    # MPT อนุญาตให้ผลรวมน้ำหนักน้อยกว่า 1 เพื่อเหลือเงินสด
    mu = np.asarray(mu, dtype=float)
    covariance = np.asarray(covariance, dtype=float)
    n = len(mu)
    objective = lambda w: -float(w @ mu - risk_aversion * (w @ covariance @ w))
    opt = minimize(objective, np.zeros(n), method="SLSQP",
                   bounds=[(0, cap)] * n,
                   constraints=[{"type": "ineq", "fun": lambda w: 1.0 - np.sum(w)}],
                   options={"maxiter": 500, "ftol": 1e-12})
    if not opt.success:
        raise RuntimeError(f"MPT optimizer failed: {opt.message}")
    weights = pd.Series(opt.x, dtype=float)
    if (weights < -1e-8).any() or (weights > cap + 1e-8).any() or weights.sum() > 1 + 1e-8:
        raise AssertionError("MPT solution violates constraints")
    return weights


def allocation_mpt(symbols, decision_date):
    history = get_complete_history(symbols, decision_date, SHORT_WINDOW, MIN_SHORT_HISTORY)
    if history is None:
        return finalize_allocation(equal_weights(symbols)), "EW_FALLBACK"
    covariance = ledoit_wolf_cov(history).values
    mu = history.mean().values
    try:
        weights = optimize_mpt(mu, covariance)
    except RuntimeError:
        return finalize_allocation(equal_weights(symbols)), "EW_FALLBACK"
    weights.index = symbols
    # ไม่ Normalize น้ำหนัก MPT อีก เพราะจะทำลาย Cash Allocation ที่ optimizer เลือก
    return weights, "OK"

ALLOCATION_METHODS = {
    "EW": allocation_ew,
    "IV": allocation_iv,
    "ERC": allocation_erc,
    "HRP": allocation_hrp,
    "MinVar": allocation_minvar,
    "CVaR": allocation_cvar,
    "MPT": allocation_mpt,
}
assert list(ALLOCATION_METHODS) == ALLOCATION_NAMES


## 14. Regression Checks — Allocation Cap และ MPT Cash

In [146]:
# ทดสอบ Cap ของทุก Allocation ด้วยประวัติผลตอบแทนจำลองที่ใช้เฉพาะ Regression Test
_previous_execution_returns = execution_returns
_test_rng = np.random.default_rng(20261115)
_test_index = pd.bdate_range("2024-01-01", periods=300)
execution_returns = pd.DataFrame(
    {"ONE": _test_rng.normal(0.0003, 0.01, 300)},
    index=_test_index,
)
_test_decision_date = _test_index[-1] + pd.Timedelta(days=1)

# ตรวจสอบทุกวิธีในกรณีมี Buy เพียงหุ้นเดียว ซึ่งเคยเป็นจุดที่ข้าม Cap
for method_name, method in ALLOCATION_METHODS.items():
    test_weights, test_status = method(["ONE"], _test_decision_date)
    assert len(test_weights) == 1
    assert float(test_weights.iloc[0]) <= MAX_WEIGHT + 1e-10, (
        f"{method_name} ละเมิด Cap ในกรณีหุ้นเดียว: {test_weights.to_dict()}"
    )
    assert float(test_weights.sum()) <= 1 + 1e-10

# ตรวจสอบกรณีหลายหุ้นและน้ำหนักกระจุกตัว
cap_cases = [
    pd.Series({"A": 1.0}),
    pd.Series({"A": 0.8, "B": 0.2}),
    pd.Series({"A": 0.5, "B": 0.3, "C": 0.2}),
    pd.Series({"A": 0.4, "B": 0.3, "C": 0.2, "D": 0.1}),
]
for raw in cap_cases:
    capped = apply_weight_cap(raw)
    assert (capped <= MAX_WEIGHT + 1e-10).all()
    assert capped.sum() <= 1 + 1e-10

single_weight = apply_weight_cap(pd.Series({"ONE": 1.0}))
assert math.isclose(float(single_weight.loc["ONE"]), 0.25, abs_tol=1e-12)

# ผลตอบแทนคาดหวังติดลบทั้งหมด: MPT ต้องสามารถเลือกถือเงินสดทั้งหมดได้
negative_mu = np.array([-0.01, -0.02])
positive_definite_cov = np.eye(2) * 0.0001
mpt_cash_test = optimize_mpt(negative_mu, positive_definite_cov)
assert mpt_cash_test.sum() <= 1 + 1e-10
assert mpt_cash_test.sum() < 1e-8

# คืนค่า Execution Return Matrix ของงานจริงหลังจบการทดสอบ
execution_returns = _previous_execution_returns

ENGINE_REGRESSION_RESULTS["allocation_cap_25pct"] = True
ENGINE_REGRESSION_RESULTS["mpt_cash_allowed"] = True
print("Allocation cap and MPT cash regression checks: PASS")


Allocation cap and MPT cash regression checks: PASS


## 15. Order Sizing และ Board-Lot Rules

In [147]:
def calculate_buy_shares(budget, execution_price, board_lot=BOARD_LOT,
                         fee_rate=BUY_COST_RATE, slippage_rate=SLIPPAGE_RATE):
    # คำนวณจำนวนหุ้นโดยปัดลงตาม Board Lot และรวมต้นทุนในงบ
    if budget <= 0 or execution_price <= 0 or board_lot <= 0:
        return 0
    all_in_unit_cost = execution_price * (1 + fee_rate + slippage_rate)
    lots = math.floor(budget / (all_in_unit_cost * board_lot))
    return int(max(0, lots) * board_lot)


def calculate_sell_cost(gross_value):
    return gross_value * (SELL_COST_RATE + SLIPPAGE_RATE)


def calculate_buy_cost(gross_value):
    return gross_value * (BUY_COST_RATE + SLIPPAGE_RATE)

print("Order sizing helpers ready")


Order sizing helpers ready


## 16. Regression Checks — Order Sizing และ Board Lots

In [148]:
# ทดสอบว่าจำนวนหุ้นเป็น Board Lot และไม่ใช้เงินเกินงบ
_test_budget = 250_000.0
_test_price = 18.45
_test_shares = calculate_buy_shares(_test_budget, _test_price)
_test_gross = _test_shares * _test_price
_test_all_in = _test_gross * (1 + BUY_COST_RATE + SLIPPAGE_RATE)
assert _test_shares % BOARD_LOT == 0
assert _test_all_in <= _test_budget + 1e-8
assert calculate_buy_shares(1.0, _test_price) == 0
assert calculate_buy_shares(250_000.0, 0) == 0

# ทดสอบว่าการคิดต้นทุนซื้อและขายไม่เป็นค่าติดลบ
assert calculate_buy_cost(_test_gross) >= 0
assert calculate_sell_cost(_test_gross) >= 0

print({"shares": _test_shares, "gross": _test_gross, "all_in": _test_all_in})
ENGINE_REGRESSION_RESULTS["order_sizing_and_board_lot"] = True
ENGINE_REGRESSION_RESULTS["transaction_cost_formulas"] = True
print("Order sizing and board-lot regression checks: PASS")


{'shares': 13500, 'gross': 249075.0, 'all_in': 249742.4960925}
Order sizing and board-lot regression checks: PASS


## 17. Execution Price Lookup และ Daily Execution

In [149]:
def get_execution_price(date, symbol, column):
    # อ่านราคาเฉพาะจาก Execution-Price Candidate เท่านั้น
    if execution_prices is None:
        return np.nan
    row = execution_lookup.get((pd.Timestamp(date), str(symbol).upper()))
    if row is None:
        return np.nan
    if hasattr(row, "Trading_Eligible") and not bool(row.Trading_Eligible):
        return np.nan
    value = getattr(row, column, np.nan)
    return float(value) if pd.notna(value) else np.nan


def execute_one_day(decision_date, execution_date, prediction, strategy_name,
                    allocation_name, cash, holdings):
    cash = float(cash)
    holdings = holdings.copy()
    transactions = []
    blocked_orders = []
    actions = attach_actions(prediction, strategy_name).set_index("Symbol")["Action"].to_dict()
    execution_members = membership_by_date.get(pd.Timestamp(execution_date), set())

    # ขายก่อน: สัญญาณ Sell และบังคับขายเมื่อออกจาก SET50
    sell_names = set()
    for symbol, shares in holdings.items():
        if shares > 0 and (symbol not in execution_members or actions.get(symbol) == "Sell"):
            sell_names.add(symbol)
    for symbol in sorted(sell_names):
        shares = holdings.get(symbol, 0)
        price = get_execution_price(execution_date, symbol, "Open")
        if not np.isfinite(price) or price <= 0:
            blocked_orders.append({"Date": execution_date, "Symbol": symbol, "Reason": "MISSING_EXECUTION_OPEN", "Side": "SELL"})
            continue
        gross = shares * price
        cost = calculate_sell_cost(gross)
        cash += gross - cost
        holdings[symbol] = 0
        transactions.append({"Date": execution_date, "Symbol": symbol, "Side": "SELL", "Shares": shares,
                             "Price": price, "Gross": gross, "Cost": cost})

    # ซื้อหลัง Sell โดยใช้เฉพาะเงินสดคงเหลือ
    buy_names = []
    for symbol, action in sorted(actions.items()):
        if action != "Buy" or symbol not in execution_members:
            continue
        price = get_execution_price(execution_date, symbol, "Open")
        if not np.isfinite(price) or price <= 0:
            blocked_orders.append({"Date": execution_date, "Symbol": symbol, "Reason": "MISSING_EXECUTION_OPEN", "Side": "BUY"})
            continue
        buy_names.append(symbol)
    if buy_names and cash > 0:
        weights, alloc_status = ALLOCATION_METHODS[allocation_name](buy_names, decision_date)
        weights = weights.reindex(buy_names).fillna(0)
        assert (weights <= MAX_WEIGHT + 1e-8).all(), "Allocation เกิน 25%"
        assert weights.sum() <= 1 + 1e-8, "Allocation รวมเกิน 100%"
        cash_before = cash
        for symbol in buy_names:
            budget = cash_before * float(weights.get(symbol, 0))
            price = get_execution_price(execution_date, symbol, "Open")
            shares = calculate_buy_shares(budget, price)
            if shares <= 0:
                continue
            gross = shares * price
            cost = calculate_buy_cost(gross)
            if gross + cost > cash + 1e-8:
                blocked_orders.append({"Date": execution_date, "Symbol": symbol, "Reason": "INSUFFICIENT_CASH", "Side": "BUY"})
                continue
            cash -= gross + cost
            holdings[symbol] = holdings.get(symbol, 0) + shares
            transactions.append({"Date": execution_date, "Symbol": symbol, "Side": "BUY", "Shares": shares,
                                 "Price": price, "Gross": gross, "Cost": cost,
                                 "Allocation_Status": alloc_status, "Budget": budget})

    holdings = {s: int(q) for s, q in holdings.items() if q > 0}
    return cash, holdings, transactions, blocked_orders


## 18. Mark-to-Market พร้อม Stale-Price Audit

In [150]:
def mark_to_market(date, cash, holdings):
    if execution_prices is None:
        raise RuntimeError("ไม่มี Execution-Price Candidate จึงประเมินมูลค่าพอร์ตไม่ได้")
    market_value = 0.0
    valuation_records = []
    for symbol, shares in holdings.items():
        if shares <= 0:
            continue
        if symbol == "BANPU" and pd.Timestamp(date) >= BANPU_SUSPENSION_START:
            raise ValueError(
                "BANPU position is corporate-action pending; merger identity and settlement are not verified"
            )
        hist = execution_prices[
            (execution_prices["Symbol"] == symbol) &
            (execution_prices["Date"] <= pd.Timestamp(date)) &
            execution_prices["Close"].notna() &
            (execution_prices["Close"] > 0)
        ].copy()
        if "Trading_Eligible" in hist.columns:
            hist = hist.loc[hist["Trading_Eligible"].fillna(False)]
        elif "Volume" in hist.columns:
            hist = hist.loc[pd.to_numeric(hist["Volume"], errors="coerce").gt(0)]
        hist = hist.sort_values("Date")
        if hist.empty:
            raise ValueError(f"ไม่มีประวัติ Execution Close สำหรับ {symbol}")
        latest = hist.iloc[-1]
        price_date = pd.Timestamp(latest["Date"])
        stale_days = (pd.Timestamp(date) - price_date).days
        if stale_days > MAX_STALE_CALENDAR_DAYS:
            raise ValueError(
                f"Stale price เกิน {MAX_STALE_CALENDAR_DAYS} วัน: {symbol}, "
                f"price_date={price_date.date()}, valuation_date={pd.Timestamp(date).date()}, "
                f"stale_days={stale_days}"
            )
        price = float(latest["Close"])
        value = int(shares) * price
        market_value += value
        valuation_records.append({
            "Date": pd.Timestamp(date), "Symbol": symbol, "Shares": int(shares),
            "Valuation_Price": price, "Price_Date": price_date,
            "Stale_Calendar_Days": stale_days, "Is_Stale": stale_days > 0,
            "Market_Value": value,
        })
    return float(cash + market_value), float(market_value), pd.DataFrame(valuation_records)


## 19. Backtest Engine (เรียกใช้ได้เมื่อ Gate เปิดเท่านั้น)

In [151]:
def run_backtest(strategy_name, allocation_name):
    if not BACKTEST_GATE_OPEN:
        raise RuntimeError("Backtest Gate ยังปิดอยู่: ห้ามรัน 63 Experiments")
    if feature_master is None or model is None or execution_prices is None:
        raise RuntimeError("ข้อมูล Feature, Model หรือ Execution Candidate ไม่ครบ")

    cash, holdings = INITIAL_CASH, {}
    daily_records, transactions, blocked_orders, valuation_audit, allocation_audit = [], [], [], [], []
    for decision_date in feature_test_dates:
        if decision_date not in next_feature_date:
            continue
        execution_date = next_feature_date[decision_date]
        prediction = predict_for_date(decision_date)
        cash, holdings, day_tx, day_blocked = execute_one_day(
            decision_date, execution_date, prediction, strategy_name,
            allocation_name, cash, holdings
        )
        for transaction in day_tx:
            transaction["Strategy"] = strategy_name
            transaction["Allocation"] = allocation_name
        for blocked in day_blocked:
            blocked["Strategy"] = strategy_name
            blocked["Allocation"] = allocation_name
        transactions.extend(day_tx)
        blocked_orders.extend(day_blocked)
        equity, market_value, valuation = mark_to_market(execution_date, cash, holdings)
        if not valuation.empty:
            valuation["Strategy"] = strategy_name
            valuation["Allocation"] = allocation_name
            valuation_audit.append(valuation)
            for row in valuation.itertuples(index=False):
                allocation_audit.append({
                    "Date": pd.Timestamp(execution_date), "Strategy": strategy_name,
                    "Allocation": allocation_name, "Symbol": row.Symbol, "Shares": int(row.Shares),
                    "Market_Value": float(row.Market_Value),
                    "Portfolio_Weight": float(row.Market_Value / equity) if equity > 0 else np.nan,
                })
        allocation_audit.append({
            "Date": pd.Timestamp(execution_date), "Strategy": strategy_name,
            "Allocation": allocation_name, "Symbol": "CASH", "Shares": np.nan,
            "Market_Value": float(cash),
            "Portfolio_Weight": float(cash / equity) if equity > 0 else np.nan,
        })
        if equity > 0:
            day_weights = [row["Portfolio_Weight"] for row in allocation_audit if row["Date"] == pd.Timestamp(execution_date)]
            assert math.isclose(sum(day_weights), 1.0, rel_tol=0, abs_tol=1e-10), "Daily realized allocation weights do not sum to 1"
        daily_records.append({
            "Date": execution_date, "Decision_Date": decision_date,
            "Strategy": strategy_name, "Allocation": allocation_name,
            "Cash": cash, "Market_Value": market_value,
            "Portfolio_Value": equity, "Holdings_Count": len(holdings),
        })
    return {
        "daily": pd.DataFrame(daily_records),
        "transactions": pd.DataFrame(transactions),
        "blocked_orders": pd.DataFrame(blocked_orders),
        "valuation_audit": pd.concat(valuation_audit, ignore_index=True) if valuation_audit else pd.DataFrame(),
        "allocation_audit": pd.DataFrame(allocation_audit),
    }


## 20. Performance Metrics — รวม Initial Equity

In [152]:
def calculate_performance(daily_curve, initial_equity=INITIAL_CASH, initial_date=INITIAL_DATE):
    # คำนวณผลตอบแทนโดยรวม Initial Equity เป็นจุดแรกของ Equity Curve
    initial_equity = float(initial_equity)
    initial_date = pd.Timestamp(initial_date)
    if daily_curve is None or daily_curve.empty:
        return {
            "Total_Return": 0.0, "CAGR": 0.0, "Annualized_Volatility": 0.0,
            "Sharpe": np.nan, "Sortino": np.nan, "Max_Drawdown": 0.0,
            "Calmar": np.nan,
        }
    data = daily_curve[["Date", "Portfolio_Value"]].copy()
    data["Date"] = pd.to_datetime(data["Date"]).dt.normalize()
    data = data.sort_values("Date").drop_duplicates("Date", keep="last")
    initial_row = pd.DataFrame({"Date": [initial_date], "Portfolio_Value": [initial_equity]})
    curve = pd.concat([initial_row, data], ignore_index=True)
    curve = curve.sort_values("Date").drop_duplicates("Date", keep="last")
    values = curve["Portfolio_Value"].astype(float)
    final_value = float(values.iloc[-1])
    total_return = final_value / initial_equity - 1
    elapsed_days = (curve["Date"].iloc[-1] - initial_date).days
    years = elapsed_days / 365.25 if elapsed_days > 0 else np.nan
    cagr = (final_value / initial_equity) ** (1 / years) - 1 if years and years > 0 and final_value > 0 else np.nan
    returns = values.pct_change().dropna()
    daily_std = returns.std(ddof=1)
    ann_vol = daily_std * np.sqrt(252) if pd.notna(daily_std) else 0.0
    sharpe = returns.mean() / daily_std * np.sqrt(252) if pd.notna(daily_std) and daily_std > 0 else np.nan
    downside = returns[returns < 0]
    downside_std = np.sqrt(np.mean(np.square(downside))) if len(downside) else np.nan
    sortino = returns.mean() / downside_std * np.sqrt(252) if pd.notna(downside_std) and downside_std > 0 else np.nan
    drawdown = values / values.cummax() - 1
    max_dd = float(drawdown.min())
    calmar = cagr / abs(max_dd) if pd.notna(cagr) and max_dd < 0 else np.nan
    return {
        "Total_Return": total_return,
        "CAGR": cagr,
        "Annualized_Volatility": ann_vol,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "Max_Drawdown": max_dd,
        "Calmar": calmar,
    }


## 21. Regression Checks — Metrics

In [153]:
# Test fixture ที่ใช้ตรวจสอบสูตร Metrics เท่านั้น ไม่ใช่ผล Backtest
_metric_fixture = pd.DataFrame({
    "Date": [pd.Timestamp("2026-01-06"), pd.Timestamp("2026-11-13")],
    "Portfolio_Value": [1_023_009.79, 1_742_636.42],
})
_metric_result = calculate_performance(
    _metric_fixture,
    initial_equity=1_000_000.0,
    initial_date=pd.Timestamp("2026-01-05"),
)

assert math.isclose(_metric_result["Total_Return"], 0.74263642, abs_tol=1e-10)
assert math.isclose(_metric_result["CAGR"], 0.9159064109976529, abs_tol=1e-10)
print("Regression metrics:", _metric_result)
print("Total Return: 74.26% | CAGR: 91.59%")
# ทดสอบว่า Drawdown เริ่มจาก Initial Equity และสะท้อนการลดลงจากจุดสูงสุด
_drawdown_fixture = pd.DataFrame({
    "Date": [pd.Timestamp("2026-01-06"), pd.Timestamp("2026-01-07"), pd.Timestamp("2026-01-08")],
    "Portfolio_Value": [1_100_000.0, 900_000.0, 1_200_000.0],
})
_drawdown_result = calculate_performance(
    _drawdown_fixture, initial_equity=1_000_000.0,
    initial_date=pd.Timestamp("2026-01-05")
)
assert math.isclose(_drawdown_result["Max_Drawdown"], (900_000 / 1_100_000) - 1, abs_tol=1e-12)
ENGINE_REGRESSION_RESULTS["initial_equity_and_drawdown"] = True
print("Metrics regression checks: PASS")


Regression metrics: {'Total_Return': 0.74263642, 'CAGR': 0.9159064109976529, 'Annualized_Volatility': np.float64(7.637817003695282), 'Sharpe': np.float64(11.984150742002253), 'Sortino': nan, 'Max_Drawdown': 0.0, 'Calmar': nan}
Total Return: 74.26% | CAGR: 91.59%
Metrics regression checks: PASS


## Engine semantic regression checks
Exercise sell-first, forced exits, holds, missing-price blocking, costs, stale-price limits, and feature-only model inputs on isolated fixtures. These tests do not certify data-source integrity.

In [154]:
_saved_engine_state = {
    "execution_prices": execution_prices, "execution_lookup": execution_lookup,
    "execution_returns": execution_returns, "membership_by_date": membership_by_date,
    "allocation_methods": ALLOCATION_METHODS,
}
try:
    _test_execution_date = pd.Timestamp("2026-03-10")
    _test_decision_date = pd.Timestamp("2026-03-09")
    _test_prices = pd.DataFrame([
        {"Date": _test_execution_date, "Symbol": "A", "Open": 100.0, "High": 100.0, "Low": 100.0, "Close": 100.0},
        {"Date": _test_execution_date, "Symbol": "B", "Open": 10.0, "High": 10.0, "Low": 10.0, "Close": 10.0},
        {"Date": _test_execution_date, "Symbol": "C", "Open": 10.0, "High": 10.0, "Low": 10.0, "Close": 10.0},
        {"Date": _test_execution_date, "Symbol": "D", "Open": np.nan, "High": 10.0, "Low": 10.0, "Close": 10.0},
    ])
    execution_prices = _test_prices
    execution_lookup = {(row.Date, row.Symbol): row for row in _test_prices.itertuples(index=False)}
    execution_returns = pd.DataFrame()
    membership_by_date = {_test_execution_date: {"A", "B", "C", "D"}}
    ALLOCATION_METHODS = dict(ALLOCATION_METHODS)
    ALLOCATION_METHODS["EW"] = lambda symbols, date: (apply_weight_cap(pd.Series(1.0, index=symbols)), "TEST")
    _test_prediction = pd.DataFrame({"Symbol": ["A", "B", "C", "D"], "Predicted_Class": [1, 0, 2, 0]})
    _test_cash, _test_holdings, _test_transactions, _test_blocked = execute_one_day(
        _test_decision_date, _test_execution_date, _test_prediction, "S2", "EW", 0.0, {"A": 100, "C": 100}
    )
    assert [row["Side"] for row in _test_transactions] == ["SELL", "BUY"]
    assert [_test_transactions[0]["Symbol"], _test_transactions[1]["Symbol"]] == ["A", "B"]
    assert _test_holdings == {"B": 200, "C": 100}
    assert any(row["Symbol"] == "D" and row["Side"] == "BUY" and row["Reason"] == "MISSING_EXECUTION_OPEN" for row in _test_blocked)
    assert _test_cash >= 0
    assert math.isclose(_test_transactions[0]["Cost"], _test_transactions[0]["Gross"] * (SELL_COST_RATE + SLIPPAGE_RATE), abs_tol=1e-10)
    assert math.isclose(_test_transactions[1]["Cost"], _test_transactions[1]["Gross"] * (BUY_COST_RATE + SLIPPAGE_RATE), abs_tol=1e-10)
    _test_equity, _test_market_value, _test_valuation = mark_to_market(_test_execution_date, _test_cash, _test_holdings)
    assert math.isclose(_test_market_value, 200 * 10.0 + 100 * 10.0, abs_tol=1e-10)
    assert math.isclose(_test_equity, _test_cash + 3000.0, abs_tol=1e-10)

    membership_by_date[_test_execution_date] = {"B"}
    _, _forced_holdings, _forced_transactions, _ = execute_one_day(
        _test_decision_date, _test_execution_date,
        pd.DataFrame({"Symbol": ["C", "B"], "Predicted_Class": [2, 2]}),
        "S2", "EW", 0.0, {"C": 100},
    )
    assert "C" not in _forced_holdings and _forced_transactions[0]["Side"] == "SELL"

    membership_by_date[_test_execution_date] = {"C"}
    _, _holdings, _hold_transactions, _ = execute_one_day(
        _test_decision_date, _test_execution_date,
        pd.DataFrame({"Symbol": ["C"], "Predicted_Class": [2]}),
        "S2", "EW", 25.0, {"C": 100},
    )
    assert _holdings == {"C": 100} and not _hold_transactions

    membership_by_date[_test_execution_date] = set()
    _, _blocked_holdings, _, _blocked_exit = execute_one_day(
        _test_decision_date, _test_execution_date,
        pd.DataFrame({"Symbol": ["D"], "Predicted_Class": [2]}),
        "S2", "EW", 0.0, {"D": 100},
    )
    assert _blocked_holdings == {"D": 100}
    assert any(row["Side"] == "SELL" and row["Reason"] == "MISSING_EXECUTION_OPEN" for row in _blocked_exit)

    _stale_date = pd.Timestamp("2026-03-10")
    execution_prices = pd.DataFrame([{
        "Date": _stale_date - pd.Timedelta(days=MAX_STALE_CALENDAR_DAYS + 1),
        "Symbol": "STALE", "Close": 10.0,
    }])
    try:
        mark_to_market(_stale_date, 0.0, {"STALE": 100})
    except ValueError as exc:
        assert "Stale price" in str(exc)
    else:
        raise AssertionError("Stale valuation over the configured limit was not rejected")
finally:
    execution_prices = _saved_engine_state["execution_prices"]
    execution_lookup = _saved_engine_state["execution_lookup"]
    execution_returns = _saved_engine_state["execution_returns"]
    membership_by_date = _saved_engine_state["membership_by_date"]
    ALLOCATION_METHODS = _saved_engine_state["allocation_methods"]

class _FeatureOnlySpyModel:
    classes_ = np.asarray(CLASS_ORDER)
    def predict(self, frame):
        self.seen_columns = list(frame.columns)
        self.seen_symbols = list(frame.index)
        return np.full(len(frame), 2)
    def predict_proba(self, frame):
        return np.tile(np.asarray([[0.1, 0.2, 0.7]]), (len(frame), 1))

_saved_prediction_state = (feature_master, model, daily_feature_frames, membership_by_date)
try:
    _prediction_date = pd.Timestamp("2026-03-09")
    _feature_row = {column: 0.0 for column in FEATURES}
    _feature_row["Target"] = 1
    feature_master = pd.DataFrame()
    model = _FeatureOnlySpyModel()
    daily_feature_frames = {_prediction_date: pd.DataFrame([_feature_row], index=pd.Index(["TEST"], name="Symbol"))}
    membership_by_date = {_prediction_date: {"TEST"}}
    _ = predict_for_date(_prediction_date)
    assert model.seen_columns == FEATURES and "Target" not in model.seen_columns

    _post_merger_date = pd.Timestamp("2026-08-10")
    _post_merger_frame = pd.DataFrame(
        [_feature_row, _feature_row], index=pd.Index(["BANPU", "TEST"], name="Symbol")
    )
    daily_feature_frames = {_post_merger_date: _post_merger_frame}
    membership_by_date = {_post_merger_date: {"BANPU", "TEST"}}
    _post_merger_prediction = predict_for_date(_post_merger_date)
    assert "BANPU" not in set(_post_merger_prediction["Symbol"])
finally:
    feature_master, model, daily_feature_frames, membership_by_date = _saved_prediction_state

# Data integrity and security-identity tests use only the frozen local snapshot.
assert not YFINANCE_REDOWNLOAD_PERFORMED
assert not ORIGINAL_SNAPSHOT_MODIFIED
assert sha256_file(EXECUTION_SNAPSHOT_PATH) == EXPECTED_ORIGINAL_SNAPSHOT_SHA256
assert GLOBAL_ADJUSTED_ROW_COUNT == 36
assert GLOBAL_RAW_TICK_AUDIT_MISMATCH_COUNT == 141
assert GLOBAL_TICK_DIAGNOSTIC_SUMMARY["diagnostic_matches_after_rescale"] == 141
assert GLOBAL_TICK_AUDIT_MISMATCH_COUNT == 0  # 2e-6 arithmetic tolerance only; no prices are rounded.
assert GLOBAL_VOLUME_UNCHANGED and GLOBAL_ADJ_CLOSE_UNCHANGED
assert security_identity_for("BANPU", pd.Timestamp("2026-07-16")) == "BANPU_OLD"
assert security_identity_for("BANPU", pd.Timestamp("2026-08-04")) == "BANPU_NEW"
assert not bool(execution_lookup[(pd.Timestamp("2026-07-20"), "BANPU")].Trading_Eligible)
assert not np.isfinite(get_execution_price(pd.Timestamp("2026-08-04"), "BANPU", "Open"))
try:
    mark_to_market(pd.Timestamp("2026-08-05"), 0.0, {"BANPU": 100})
except ValueError as exc:
    assert "corporate-action pending" in str(exc)
else:
    raise AssertionError("BANPU merger position was valued without verified identity/settlement")
try:
    require_reconciled_corporate_actions(corporate_action_ledger)
except ValueError as exc:
    assert "cannot be applied" in str(exc)
else:
    raise AssertionError("Unverified corporate actions were accepted for accounting")

ENGINE_REGRESSION_RESULTS.update({
    "snapshot_hash_unchanged_and_no_redownload": True,
    "global_basis_adjustment_isolated_and_not_rounded": True,
    "banpu_suspended_and_new_identity_bars_blocked": True,
    "banpu_post_merger_features_excluded": True,
    "unverified_corporate_actions_rejected": True,
    "sell_before_buy_and_available_cash": True,
    "forced_exit_and_hold": True,
    "missing_open_orders_are_audited": True,
    "transaction_costs_both_sides": True,
    "daily_equity_cash_plus_positions": True,
    "stale_valuation_is_bounded": True,
    "prediction_uses_features_only": True,
})
print("Engine semantic regression checks: PASS")


Engine semantic regression checks: PASS


## 22. Gate Audit — เหตุผลที่ยังห้ามรัน 63 ชุด

In [155]:
ENGINE_REQUIRED_CHECKS = [
    "snapshot_hash_unchanged_and_no_redownload",
    "global_basis_adjustment_isolated_and_not_rounded",
    "banpu_suspended_and_new_identity_bars_blocked",
    "banpu_post_merger_features_excluded",
    "unverified_corporate_actions_rejected",
    "allocation_cap_25pct", "mpt_cash_allowed", "order_sizing_and_board_lot",
    "transaction_cost_formulas", "initial_equity_and_drawdown",
    "sell_before_buy_and_available_cash", "forced_exit_and_hold",
    "missing_open_orders_are_audited", "transaction_costs_both_sides",
    "daily_equity_cash_plus_positions", "stale_valuation_is_bounded",
    "prediction_uses_features_only",
]
ENGINE_INTEGRITY_PASSED = all(
    ENGINE_REGRESSION_RESULTS.get(name) is True for name in ENGINE_REQUIRED_CHECKS
)
metadata_source = str(original_execution_metadata.get("source", "UNKNOWN"))
source_is_official = bool(
    VERIFIED_EXECUTION_PRICE_PATH.is_file()
    and verified_data_status.get("verified_dataset_created") is True
    and verified_data_status.get("source_approved") is True
)
execution_prices_complete = (
    price_integrity_summary["missing_price_values"] == 0
    and price_integrity_summary["nonpositive_price_values"] == 0
    and price_integrity_summary["invalid_ohlc_rows"] == 0
    and price_integrity_summary["duplicate_date_symbol_keys"] == 0
)
SNAPSHOT_INTEGRITY_CONFIRMED = (
    sha256_file(EXECUTION_SNAPSHOT_PATH) == EXPECTED_ORIGINAL_SNAPSHOT_SHA256
    and sha256_file(EXECUTION_ORIGINAL_COPY_PATH) == EXPECTED_ORIGINAL_SNAPSHOT_SHA256
    and not ORIGINAL_SNAPSHOT_MODIFIED
)
GLOBAL_PRICE_BASIS_CANDIDATE_CREATED = (
    GLOBAL_ADJUSTED_ROW_COUNT == 36
    and GLOBAL_VOLUME_UNCHANGED
    and GLOBAL_ADJ_CLOSE_UNCHANGED
    and sha256_file(EXECUTION_CORRECTED_PATH)
    == execution_data_lineage["derived_candidate"]["sha256"]
)
ledger_sources_present = (
    corporate_action_ledger["Official_Source"].fillna("").astype(str).str.startswith("https://")
)
ledger_all_reconciled = corporate_action_ledger["Reconciliation_Status"].eq("PASS").all()
corporate_action_accounting_implemented = False
CORPORATE_ACTIONS_CONFIRMED = bool(
    ledger_sources_present.all() and ledger_all_reconciled and corporate_action_accounting_implemented
)
BANPU_MISSING_DATA_RESOLVED = bool(
    len(banpu_missing_audit) == 0
    and banpu_missing_summary["official_set_calendar_verified"]
    and banpu_missing_summary["security_identity_verified"]
    and banpu_missing_summary["new_identity_features_rebuilt"]
)
TICK_AUDIT_PASSED = bool(source_is_official and tick_audit.empty)
EXECUTION_PRICES_CONFIRMED = bool(
    source_is_official and execution_prices_complete
    and execution_manifest_complete and TICK_AUDIT_PASSED
)

GATE_REQUIREMENTS = {
    "Frozen snapshot hash and immutable copy": (
        "Hash the frozen input and archived byte-for-byte copy; compare both with the pinned SHA-256.",
        "Both hashes match the frozen snapshot and the source file is unchanged.",
    ),
    "No Yahoo download was performed": (
        "Inspect the notebook execution path and run flag.",
        "No yfinance download call runs; the existing snapshot is the only Yahoo input.",
    ),
    "Original request manifest is complete": (
        "Read the original sidecar without inferring missing values.",
        "Interval, grouping, and requested ticker list are explicitly recorded; UNKNOWN is not a pass.",
    ),
    "GLOBAL corrected candidate is reproducible": (
        "Apply 28/27 only to GLOBAL OHLC before 2026-02-24; compare hashes and unchanged columns.",
        "A separate candidate exists; only declared OHLC cells change; this is not exchange verification.",
    ),
    "Authorized execution data is available": (
        "Require a verified SET/authorized price dataset and inspect its source manifest.",
        "The selected dataset is source-verified and covers required execution and valuation dates.",
    ),
    "Vendor action inventory is retained": (
        "Preserve every non-zero vendor action row and its source label.",
        "Every vendor event is retained and marked unreconciled until official terms are attached.",
    ),
    "Corporate actions reconciled and accounted": (
        "Reconcile every ledger row to an official URL/document and run integrated portfolio accounting tests.",
        "All events pass source reconciliation, are applied once at correct dates, and dividend/split/merger tests pass.",
    ),
    "BANPU identity, trading status, and missing prices resolved": (
        "Use SET calendar/status and source price patches keyed by Date and Security_ID; reject stale/zero-volume bars.",
        "Suspension, old/new identity, conversion fractions, all price gaps, and position valuation are resolved without forward fill.",
    ),
    "BANPU new-identity features verified": (
        "Rebuild six model features from post-2026-08-04 data under the unchanged training pipeline.",
        "Only verified new-identity feature history is used; old/new security history is not concatenated.",
    ),
    "Corrected candidate tick audit passes": (
        "Re-audit all OHLC on the corrected candidate with date-effective ticks and retain strict residuals.",
        "No unresolved tick exceptions remain; numerical tolerance does not round or alter prices.",
    ),
    "Focused engine regression suite": (
        "Run fixtures for data eligibility, order behavior, allocation, fees, valuation, and prediction inputs.",
        "Every required engine regression key is True.",
    ),
    "All data and engine gates pass": (
        "Aggregate all snapshot, price, corporate-action, BANPU, tick, and engine checks.",
        "Every prerequisite is PASS and the output contract is validated.",
    ),
}


def gate_row(gate, check, status, observed, evidence):
    method, criteria = GATE_REQUIREMENTS[check]
    return {
        "Gate": gate, "Check": check, "Method": method, "Pass_Criteria": criteria,
        "Status": status, "Observed": str(observed), "Evidence": str(evidence),
    }


research_gate_checklist = [
    gate_row("Snapshot Integrity", "Frozen snapshot hash and immutable copy",
             "PASS" if SNAPSHOT_INTEGRITY_CONFIRMED else "FAIL",
             f"source_sha256={original_hash_before}; original_modified={ORIGINAL_SNAPSHOT_MODIFIED}",
             EXECUTION_DATA_LINEAGE_PATH),
    gate_row("Execution Prices", "No Yahoo download was performed",
             "PASS" if not YFINANCE_REDOWNLOAD_PERFORMED else "FAIL",
             f"yfinance_redownload_performed={YFINANCE_REDOWNLOAD_PERFORMED}",
             EXECUTION_DATA_LINEAGE_PATH),
    gate_row("Execution Prices", "Original request manifest is complete",
             "PASS" if execution_manifest_complete else "BLOCKED",
             f"interval={original_execution_metadata.get('interval', 'UNKNOWN')}; "
             f"group_by={original_execution_metadata.get('group_by', 'UNKNOWN')}; "
             f"requested_ticker_list={'present' if original_execution_metadata.get('tickers') else 'UNKNOWN'}",
             ORIGINAL_EXECUTION_METADATA_PATH),
    gate_row("Price Basis", "GLOBAL corrected candidate is reproducible",
             "PASS" if GLOBAL_PRICE_BASIS_CANDIDATE_CREATED else "FAIL",
             f"pre_ex_rows={GLOBAL_ADJUSTED_ROW_COUNT}; factor={GLOBAL_SHARE_FACTOR}; "
             f"volume_unchanged={GLOBAL_VOLUME_UNCHANGED}; "
             f"candidate_sha256={execution_data_lineage['derived_candidate']['sha256']}",
             EXECUTION_DATA_LINEAGE_PATH),
    gate_row("Execution Prices", "Authorized execution data is available",
             "PASS" if EXECUTION_PRICES_CONFIRMED else "BLOCKED",
             f"verified_file={VERIFIED_EXECUTION_PRICE_PATH.is_file()}; "
             f"exchange_verified={price_integrity_summary['exchange_verified']}; "
             f"missing_price_values={price_integrity_summary['missing_price_values']}",
             VERIFIED_DATA_STATUS_PATH),
    gate_row("Corporate Actions", "Vendor action inventory is retained",
             "PASS" if len(corporate_action_vendor_events) > 0
             and len(vendor_ledger) == len(corporate_action_vendor_events) else "BLOCKED",
             f"vendor_events={len(corporate_action_vendor_events)}; "
             f"all_application_flags_false={not corporate_action_ledger['Applied_By_Portfolio_Engine'].fillna(False).any()}",
             CORPORATE_ACTION_LEDGER_PATH),
    gate_row("Corporate Actions", "Corporate actions reconciled and accounted",
             "PASS" if CORPORATE_ACTIONS_CONFIRMED else "BLOCKED",
             f"all_official_urls={bool(ledger_sources_present.all())}; "
             f"all_rows_reconciled={bool(ledger_all_reconciled)}; "
             f"accounting_implemented={corporate_action_accounting_implemented}",
             CORPORATE_ACTION_LEDGER_PATH),
    gate_row("BANPU", "BANPU identity, trading status, and missing prices resolved",
             "PASS" if BANPU_MISSING_DATA_RESOLVED else "BLOCKED",
             banpu_missing_summary, banpu_missing_audit_path),
    gate_row("BANPU", "BANPU new-identity features verified",
             "PASS" if BANPU_FEATURES_VERIFIED_AFTER_MERGER else "BLOCKED",
             "Post-merger BANPU features are excluded; no verified new-identity rebuild is available.",
             FEATURE_MASTER_PATH),
    gate_row("Tick Audit", "Corrected candidate tick audit passes",
             "PASS" if TICK_AUDIT_PASSED else "BLOCKED",
             f"raw_GLOBAL={GLOBAL_RAW_TICK_AUDIT_MISMATCH_COUNT}; "
             f"corrected_GLOBAL_at_2e-6={GLOBAL_TICK_AUDIT_MISMATCH_COUNT}; "
             f"strict_1e-6_aligned={int(global_tick_diagnostic['Strict_Tick_Aligned'].sum())}/141; "
             f"verified_source={source_is_official}",
             global_tick_diagnostic_path),
    gate_row("Engine Integrity", "Focused engine regression suite",
             "PASS" if ENGINE_INTEGRITY_PASSED else "FAIL",
             ENGINE_REGRESSION_RESULTS, "05_backtest.ipynb regression cells"),
]
prerequisites_pass = all(row["Status"] == "PASS" for row in research_gate_checklist)
research_gate_checklist.append(
    gate_row("Research Backtest", "All data and engine gates pass",
             "PASS" if prerequisites_pass else "BLOCKED",
             f"{sum(row['Status'] == 'PASS' for row in research_gate_checklist)} "
             f"of {len(research_gate_checklist)} prerequisite checks passed",
             OUTPUT_DIR / "research_gate_checklist.csv")
)
gate_blockers = [
    f"{row['Gate']}: {row['Check']} ({row['Observed']})"
    for row in research_gate_checklist if row["Status"] != "PASS"
]
BACKTEST_GATE_OPEN = len(gate_blockers) == 0
RESEARCH_GATE_EVIDENCE_PATH = OUTPUT_DIR / "research_gate_checklist.csv"
pd.DataFrame(
    research_gate_checklist,
    columns=["Gate", "Check", "Method", "Pass_Criteria", "Status", "Observed", "Evidence"],
).to_csv(RESEARCH_GATE_EVIDENCE_PATH, index=False)
backtest_run_status = {
    "backtest_gate_open": BACKTEST_GATE_OPEN,
    "backtest_63_sets_executed": False,
    "new_backtest_results_created": False,
    "yfinance_redownload_performed": YFINANCE_REDOWNLOAD_PERFORMED,
    "original_snapshot_modified": ORIGINAL_SNAPSHOT_MODIFIED,
    "snapshot_integrity_confirmed": SNAPSHOT_INTEGRITY_CONFIRMED,
    "global_price_basis_candidate_created": GLOBAL_PRICE_BASIS_CANDIDATE_CREATED,
    "verified_execution_data_available": VERIFIED_EXECUTION_PRICE_PATH.is_file(),
    "gate_checklist": research_gate_checklist,
    "engine_regression_checks": ENGINE_REGRESSION_RESULTS,
    "engine_integrity_passed": ENGINE_INTEGRITY_PASSED,
    "execution_price_integrity": price_integrity_summary,
    "banpu_missing_data": banpu_missing_summary,
    "corporate_action_vendor_event_count": int(len(corporate_action_vendor_events)),
    "corporate_action_ledger_rows": int(len(corporate_action_ledger)),
    "corporate_action_accounting_implemented": corporate_action_accounting_implemented,
    "global_tick_basis_diagnostic": GLOBAL_TICK_DIAGNOSTIC_SUMMARY,
    "evidence_files": {
        "checklist": str(RESEARCH_GATE_EVIDENCE_PATH),
        "lineage": str(EXECUTION_DATA_LINEAGE_PATH),
        "verified_data_status": str(VERIFIED_DATA_STATUS_PATH),
        "original_snapshot_copy": str(EXECUTION_ORIGINAL_COPY_PATH),
        "corrected_candidate": str(EXECUTION_CORRECTED_PATH),
        "corporate_action_ledger": str(CORPORATE_ACTION_LEDGER_PATH),
        "security_identity_mapping": str(SECURITY_IDENTITY_MAPPING_PATH),
        "execution_price_audit": str(execution_integrity_path),
        "vendor_actions": str(corporate_action_events_path),
        "banpu_gaps": str(banpu_missing_audit_path),
        "global_tick_audit_original": str(raw_tick_audit_path),
        "global_tick_audit_corrected": str(tick_audit_path),
        "global_tick_diagnostic": str(global_tick_diagnostic_path),
        "set_recent_crosscheck": str(set_recent_crosscheck_path),
    },
    "project_root": str(PROJECT_ROOT),
    "archive_manifest": str(ARCHIVE_MANIFEST_PATH),
    "blockers": gate_blockers,
}
print("RESEARCH BACKTEST GATE:", "OPEN" if BACKTEST_GATE_OPEN else "CLOSED")
for row in research_gate_checklist:
    print(f"{row['Status']}: {row['Gate']} - {row['Check']} - {row['Observed']}")
with STATUS_PATH.open("w", encoding="utf-8") as stream:
    json.dump(backtest_run_status, stream, ensure_ascii=False, indent=2, default=str)
print("Status saved:", STATUS_PATH.resolve())


RESEARCH BACKTEST GATE: CLOSED
PASS: Snapshot Integrity - Frozen snapshot hash and immutable copy - source_sha256=14ca82c9a481118910952cfaaaeea08daecd6548d9d99e19a66e4d9c51ef986a; original_modified=False
PASS: Execution Prices - No Yahoo download was performed - yfinance_redownload_performed=False
BLOCKED: Execution Prices - Original request manifest is complete - interval=UNKNOWN; group_by=UNKNOWN; requested_ticker_list=UNKNOWN
PASS: Price Basis - GLOBAL corrected candidate is reproducible - pre_ex_rows=36; factor=1.037037037037037; volume_unchanged=True; candidate_sha256=4d35f215ce3a0243b13ef3eb582fc7904b0de8920ad85c78ee14497ff1fc5428
BLOCKED: Execution Prices - Authorized execution data is available - verified_file=False; exchange_verified=False; missing_price_values=253
PASS: Corporate Actions - Vendor action inventory is retained - vendor_events=136; all_application_flags_false=True
BLOCKED: Corporate Actions - Corporate actions reconciled and accounted - all_official_urls=False; 

## 23. Run / Save — ไม่สร้างผลใหม่เมื่อ Gate ปิด

In [156]:
backtest_outputs = None

if BACKTEST_GATE_OPEN:
    experiment_results = []
    for strategy_name, allocation_name in product(STRATEGIES.keys(), ALLOCATION_NAMES):
        print("Running:", strategy_name, allocation_name)
        experiment_results.append(run_backtest(strategy_name, allocation_name))
    backtest_outputs = experiment_results
    expected_combinations = {(strategy, allocation) for strategy in STRATEGIES for allocation in ALLOCATION_NAMES}
    actual_combinations = {
        (result["daily"]["Strategy"].iloc[0], result["daily"]["Allocation"].iloc[0])
        for result in experiment_results if not result["daily"].empty
    }
    if len(experiment_results) != 63 or actual_combinations != expected_combinations:
        raise AssertionError(f"Expected 63 complete combinations; got {len(experiment_results)} runs and {len(actual_combinations)} nonempty combinations")
    expected_dates = None
    for result in experiment_results:
        dates = pd.DatetimeIndex(pd.to_datetime(result["daily"]["Date"]).sort_values().unique())
        if expected_dates is None:
            expected_dates = dates
        elif not dates.equals(expected_dates):
            raise AssertionError("Research combinations do not share the same daily valuation dates")

    all_daily = pd.concat([r["daily"] for r in experiment_results], ignore_index=True)
    all_transactions = pd.concat([r["transactions"] for r in experiment_results if not r["transactions"].empty], ignore_index=True) if any(not r["transactions"].empty for r in experiment_results) else pd.DataFrame(columns=["Date", "Symbol", "Side", "Shares", "Price", "Gross", "Cost", "Strategy", "Allocation"])
    all_blocked = pd.concat([r["blocked_orders"] for r in experiment_results if not r["blocked_orders"].empty], ignore_index=True) if any(not r["blocked_orders"].empty for r in experiment_results) else pd.DataFrame(columns=["Date", "Symbol", "Reason", "Side"])
    valuation_frames = [r["valuation_audit"] for r in experiment_results if not r["valuation_audit"].empty]
    all_valuation = pd.concat(valuation_frames, ignore_index=True) if valuation_frames else pd.DataFrame()
    all_allocations = pd.concat([r["allocation_audit"] for r in experiment_results], ignore_index=True)
    metrics_rows = []
    for result in experiment_results:
        daily = result["daily"]
        metrics_rows.append({
            "Strategy": daily["Strategy"].iloc[0],
            "Allocation": daily["Allocation"].iloc[0],
            **calculate_performance(daily),
        })
    metrics_summary = pd.DataFrame(metrics_rows)
    if all_transactions.empty:
        cost_report = pd.DataFrame(columns=["Strategy", "Allocation", "Side", "Transaction_Count", "Total_Gross", "Total_Cost"])
    else:
        cost_report = all_transactions.groupby(["Strategy", "Allocation", "Side"], as_index=False).agg(
            Transaction_Count=("Cost", "size"), Total_Gross=("Gross", "sum"), Total_Cost=("Cost", "sum")
        )
    output_frames = {
        "daily_portfolio_results.csv": all_daily,
        "transactions.csv": all_transactions,
        "blocked_orders.csv": all_blocked,
        "valuation_audit.csv": all_valuation,
        "daily_allocation_weights.csv": all_allocations,
        "metrics_summary.csv": metrics_summary,
        "transaction_cost_report.csv": cost_report,
    }
    for filename, frame in output_frames.items():
        frame.to_csv(OUTPUT_DIR / filename, index=False)

    def sha256_file(file_path):
        digest = hashlib.sha256()
        with Path(file_path).open("rb") as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(block)
        return digest.hexdigest()
    reproducibility_config = {
        "run_timestamp_utc": pd.Timestamp.now(tz="UTC").isoformat(),
        "combination_count": len(experiment_results),
        "valuation_dates": len(expected_dates),
        "settings": {
            "initial_cash": INITIAL_CASH, "max_weight": MAX_WEIGHT,
            "buy_cost_rate": BUY_COST_RATE, "sell_cost_rate": SELL_COST_RATE,
            "slippage_rate": SLIPPAGE_RATE, "board_lot": BOARD_LOT,
            "max_stale_calendar_days": MAX_STALE_CALENDAR_DAYS,
        },
        "execution_metadata": execution_metadata,
        "input_sha256": {str(file_path): sha256_file(file_path) for file_path in REQUIRED_INPUT_PATHS},
        "notebook_sha256": sha256_file(PROJECT_ROOT / "05_backtest.ipynb"),
        "outputs": {name: str(OUTPUT_DIR / name) for name in output_frames},
    }
    (OUTPUT_DIR / "backtest_run_config.json").write_text(
        json.dumps(reproducibility_config, ensure_ascii=False, indent=2, default=str), encoding="utf-8"
    )
    backtest_run_status["backtest_63_sets_executed"] = True
    backtest_run_status["new_backtest_results_created"] = True
    backtest_run_status["combination_count"] = len(experiment_results)
    backtest_run_status["valuation_dates"] = len(expected_dates)
    with STATUS_PATH.open("w", encoding="utf-8") as status_file:
        json.dump(backtest_run_status, status_file, ensure_ascii=False, indent=2, default=str)
else:
    print("Skipped all 63 experiments because the evidence-derived validation gate is closed.")
    print("No portfolio result CSVs were written.")

print("Notebook regression cells completed; historical Backtest ran only if every required gate passed.")


Skipped all 63 experiments because the evidence-derived validation gate is closed.
No portfolio result CSVs were written.
Notebook regression cells completed; historical Backtest ran only if every required gate passed.
